<a href="https://colab.research.google.com/github/enzoalvino00/Analitica-Empresarial-Integrada/blob/main/Unidad%20_3_Analitiica_Predictiva_y_Prescriptiva/Semana_10_Empresa_analitica/Analitica_Empresarial_Integrada_Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🏛️ **TECSUP 2026-II - ANALÍTICA EMPRESARIAL INTEGRADA**

## 🔬 **LABORATORIO DIRIGIDO N.° 10**: **Pronóstico de demanda**


| | |
|---|---|
| **Docente** | Pilar Rocío Sayán Mejía |
| **Laboratorio** | Pronóstico de demanda |
| **N.° de laboratorio** | 10 |
| **Tema** | Pronóstico de demanda y evaluación de modelos de predicción |
| **Plan curricular** | Big Data y Ciencia de Datos |
| **Código** | C28 |
| **Periodo académico** | 2026-II |
| **Duración** | 200 minutos (6:50 p. m. a 10:10 p. m., con receso de 15 minutos) |
| **Modalidad** | Grupal. Las respuestas del foro son individuales. |

---

**Estudiante:** *Enzo Gabriel Alvino Carhuaricra* &nbsp;&nbsp;&nbsp; **Sección:** **6C28** &nbsp;&nbsp;&nbsp; **Fecha:** 21/10/2026


**Dataset real:** Online Retail II — UCI Machine Learning Repository (ID 502)
**Objetivo:** construir un pronóstico semanal de ventas, seleccionar modelo y política de cantidad con validación temporal sin información futura, evaluarlos una sola vez y decidir si conviene adoptarlos.

> El notebook termina en una **decisión fundamentada** (adoptar, mantener en evaluación o no adoptar), no solo en el modelo con menor error.

## Trabajo individual y entrega

Esta práctica dirigida es **individual**. Ejecute el notebook en **su propio Google Colab**, verifique los resultados y responda con sus propias interpretaciones. No se requiere informe separado: las respuestas, el cierre ejecutivo y las conclusiones quedan en este notebook.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con los retos resueltos, a la carpeta `semana10/` de su repositorio personal de GitHub.
2. Conserve la versión en su propio Google Colab y compártala con permiso de lectura.
3. En la **actividad de la semana** presente **ambos URLs**: el enlace directo al archivo en GitHub y el enlace a su Colab.

**Identificación**
- Nombre:
- Sección:
- URL de GitHub:
- URL de Colab:
- Uso de IA declarado:

# 1. Pregunta de negocio

**¿Cuántas unidades debería planificar la empresa para las próximas semanas, qué riesgo asume si el pronóstico se equivoca y conviene adoptar el modelo?**

Se distinguen cuatro pasos que no deben confundirse:

1. **Pronóstico de ventas:** estimación de las ventas semanales.
2. **Política de cantidad:** regla que convierte el pronóstico en una cantidad objetivo semanal (pronóstico + colchón).
3. **Evaluación económica:** costo de los excedentes y faltantes que produce esa política.
4. **Decisión de adopción:** usar, mantener en evaluación o no usar el modelo, según su resultado frente al baseline.

### Diseño del laboratorio

| Elemento | Definición |
|---|---|
| Unidad temporal | Semana de lunes a domingo, identificada por la fecha del lunes |
| Semanas válidas | Solo semanas completas según la cobertura del archivo |
| Variable objetivo | Unidades vendidas (registradas) por semana de un SKU del Reino Unido |
| Test final | Últimas 8 semanas completas; se evalúa una sola vez |
| Selección del SKU | Criterios calculados solo antes del test |
| Validación | Backtesting rolling-origin con horizonte de 8 semanas dentro del entrenamiento |
| Modelos | Naïve, Naïve estacional (m = 4) y Exponential Smoothing |
| Regla de selección (única) | Candidatos con costo promedio ≤ mínimo × 1,01 (incluye exactamente el 1 %). Si el mínimo es 0, solo empatan los costos 0. Desempate: menor MAE, menor desviación del costo y menor complejidad |
| Política de colchón | Multiplicador 0; 0,5; 1 o 1,5 × MAE previo. El MAE previo de cada origen usa solo errores anteriores a ese origen |

## Actividad 1: Revisión de conceptos

| Concepto | Definición con sus propias palabras y ejemplo | Pregunta de apoyo |
|---|---|---|
| Serie temporal | Es un conjunto de datos ordenados en el tiempo. Ejemplo: las ventas de una tienda registradas cada semana. | No se pueden barajar las filas porque se pierde el orden temporal y se puede utilizar información del futuro para predecir el pasado. |
| Semana completa | Es una semana que contiene todos los días, de lunes a domingo, dentro de la cobertura del archivo. Ejemplo: una semana con registros desde el lunes hasta el domingo. | Se excluye la primera semana si está incompleta porque sus ventas podrían ser menores que las reales al faltar días de registro. |
| Tendencia y estacionalidad | La tendencia es el comportamiento general de los datos a lo largo del tiempo. La estacionalidad es un patrón que se repite en intervalos regulares. Ejemplo: las ventas crecen cada año (tendencia) y aumentan cada Navidad (estacionalidad). | No basta con que la serie suba y baje; para afirmar que existe estacionalidad, el patrón debe repetirse regularmente. |
| Baseline (Naïve y Naïve estacional) | Son modelos sencillos de referencia. Naïve repite el último valor observado; Naïve estacional repite el valor del mismo periodo del ciclo anterior. Ejemplo: repetir las ventas de la semana pasada o las de cuatro semanas atrás. | Si un modelo complejo no supera al Naïve, significa que su mayor complejidad no aporta una mejora suficiente frente a una predicción sencilla. |
| Exponential Smoothing | Es un método que utiliza observaciones anteriores y da mayor peso a los datos recientes. Ejemplo: pronosticar las ventas de la próxima semana considerando las ventas anteriores. | No, agregar más componentes no garantiza un mejor pronóstico; debe comprobarse su desempeño con validación temporal. |
| Partición temporal (train, validación, test) | Es la división de los datos según el tiempo: entrenamiento para ajustar el modelo, validación para seleccionarlo y test para la evaluación final. Ejemplo: usar semanas antiguas para entrenar y las últimas ocho semanas para probar. | El test se mira una sola vez para evitar ajustar el modelo a esos datos y obtener una evaluación engañosa. |
| Backtesting rolling-origin | Es una evaluación que repite el pronóstico desde diferentes fechas de origen, respetando el orden temporal. Ejemplo: entrenar con datos hasta una semana y pronosticar las ocho siguientes; después avanzar el origen y repetir. | Permite evaluar el modelo en distintos periodos y comprobar si su desempeño es consistente, en lugar de depender de una sola división. |
| MAE, RMSE y WAPE | Son métricas que miden los errores del pronóstico. MAE calcula el error absoluto medio; RMSE penaliza más los errores grandes; WAPE expresa el error absoluto total como proporción de la demanda total. | El MAE expresa directamente el error promedio en unidades vendidas. |
| Costo asimétrico del error | Significa que sobreestimar y subestimar las ventas pueden generar costos diferentes. Ejemplo: comprar 20 unidades de más genera costos de almacenamiento, mientras que comprar 20 de menos puede ocasionar ventas perdidas. | Dos errores de 20 unidades pueden costar distinto porque el costo de los excedentes puede ser diferente al costo de los faltantes. |
| Pronóstico, política de cantidad y evaluación económica | El pronóstico estima las ventas; la política de cantidad convierte esa estimación en una cantidad objetivo mediante un colchón; la evaluación económica calcula los costos de excedentes y faltantes. Ejemplo: pronosticar 100 unidades y establecer una cantidad objetivo de 110. | No necesariamente. La cantidad objetivo no es lo mismo que la cantidad que hay que comprar, porque también deben considerarse el inventario disponible y los pedidos que llegarán a tiempo. |

In [ ]:
# ══════════════════════════════════════════════════════════════
# INSTALACIÓN DE LIBRERÍAS
# ══════════════════════════════════════════════════════════════

## Instalación de dependencias para el análisis y modelado

%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

# Importación de módulos generales
import os  ### Permite gestionar rutas y archivos
import sys  ### Permite consultar la versión de Python
import warnings  ### Permite gestionar las advertencias del código

## Librerías para análisis y visualización
import numpy as np  ### Permite realizar operaciones numéricas
import pandas as pd  ### Permite manipular tablas y series temporales
import plotly.express as px  ### Permite crear gráficos interactivos
import plotly.graph_objects as go  ### Permite personalizar gráficos interactivos

## Librerías para pronóstico y evaluación
import sklearn  ### Permite consultar la versión de scikit-learn
import statsmodels  ### Permite consultar la versión de statsmodels
from ucimlrepo import fetch_ucirepo  ### Permite descargar el dataset desde UCI
from statsmodels.tsa.holtwinters import ExponentialSmoothing  ### Importa el modelo de suavizamiento exponencial
from sklearn.metrics import mean_absolute_error, mean_squared_error  ### Importa métricas para evaluar los errores

# Configuración de visualización
## Ajustes para mostrar tablas con mayor claridad
pd.set_option("display.max_columns", 50)  ### Permite mostrar hasta 50 columnas
pd.set_option("display.width", 160)  ### Amplía el ancho de visualización

# Registro de versiones del entorno
## Permite identificar las versiones utilizadas en el laboratorio
VERSIONES = {
    "python": sys.version.split()[0],  ### Registra la versión de Python
    "pandas": pd.__version__,  ### Registra la versión de pandas
    "numpy": np.__version__,  ### Registra la versión de NumPy
    "statsmodels": statsmodels.__version__,  ### Registra la versión de statsmodels
    "scikit-learn": sklearn.__version__  ### Registra la versión de scikit-learn
}

print("Versiones:", VERSIONES)  ### Muestra las versiones instaladas

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. DESCARGA DE DATOS REALES Y TRAZABILIDAD
# ══════════════════════════════════════════════════════════════

# Importación de herramientas para descargar y leer archivos
from io import BytesIO  ### Permite tratar datos binarios como archivos en memoria
from urllib.request import urlopen  ### Permite descargar archivos desde una URL
from zipfile import ZipFile  ### Permite abrir archivos ZIP

# Función para leer y unir las dos hojas del archivo Excel
## Se conserva el nombre de las hojas para registrar la trazabilidad
def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  ### Lee todas las hojas del archivo Excel
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)  ### Une las hojas y devuelve sus nombres

# Inicialización de variables de control
df, fuente, hojas_leidas = None, None, None  ### Inicializa los resultados antes de intentar la descarga

# Primer intento: API oficial de UCI
try:
    retail = fetch_ucirepo(id=502)  ### Recupera el dataset Online Retail II
    df = retail.data.original.copy()  ### Copia los datos originales devueltos por la API
    fuente = "API ucimlrepo (UCI ID 502)"  ### Registra la fuente utilizada
    hojas_leidas = "tabla única entregada por la API"  ### Registra el formato recibido
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)  ### Informa el tipo de error sin detener el proceso

# Segundo intento: archivo ZIP oficial de UCI
if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"  ### Define la URL del archivo ZIP
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()  ### Descarga el archivo ZIP en memoria

        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))  ### Identifica el archivo Excel
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))  ### Lee y concatena las dos hojas

        fuente = "ZIP oficial de UCI"  ### Registra la fuente utilizada
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)  ### Informa el tipo de error del segundo intento

# Tercer intento: archivo local o carga manual
if df is None:
    ## Busca archivos locales que coincidan con el nombre del dataset
    locales = sorted(
        f for f in os.listdir(".")
        if "online" in f.lower()
        and "retail" in f.lower()
        and f.lower().endswith((".xlsx", ".csv"))
    )  ### Identifica archivos Excel o CSV disponibles en la carpeta actual

    if not locales:
        try:
            from google.colab import files  ### Importa la herramienta de carga de Google Colab
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())  ### Permite seleccionar y cargar un archivo manualmente
        except ImportError:
            raise FileNotFoundError(
                "Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar."
            )  ### Solicita colocar el archivo en la carpeta de trabajo si no se usa Colab

    archivo = locales[0]  ### Selecciona el primer archivo disponible

    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)  ### Lee y concatena las hojas del Excel
    else:
        df = pd.read_csv(
            archivo,
            dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str}
        )  ### Lee el CSV y conserva como texto los identificadores
        hojas_leidas = "no aplica (CSV)"  ### Indica que el archivo no tiene hojas de Excel

    fuente = f"archivo local: {archivo}"  ### Registra el nombre del archivo utilizado

# Registro de trazabilidad y cobertura temporal
## Identificación de la columna de fecha
columnas_fecha = [c for c in df.columns if "date" in str(c).lower()]  ### Busca columnas cuyo nombre contenga date

if not columnas_fecha:
    raise ValueError("No se encontró una columna de fecha en el dataset.")  ### Detiene el proceso si falta la fecha

fechas_raw = pd.to_datetime(df[columnas_fecha[0]], errors="coerce")  ### Convierte la columna a fechas e identifica valores inválidos

print("Fuente utilizada:", fuente)  ### Muestra la procedencia de los datos
print("Hojas:", hojas_leidas)  ### Muestra las hojas leídas o el formato recibido
print("Dimensión:", df.shape)  ### Muestra el número de filas y columnas
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())  ### Muestra la cobertura temporal del dataset
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())  ### Registra la fecha y hora de ejecución en UTC
print("Versiones:", VERSIONES)  ### Muestra las versiones del entorno utilizado

display(df.head())  ### Muestra las primeras cinco filas para revisar la estructura

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-26T04:02:29.381484+00:00
Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### **Obtención y caracterización inicial del conjunto de datos Online Retail II**

La descarga del conjunto de datos se realizó correctamente mediante el archivo ZIP oficial de UCI, debido a que la API `ucimlrepo` no reconoció el identificador solicitado. Se integraron las dos hojas del archivo, obteniendo **1,067,371 registros y 8 columnas**, con transacciones comprendidas entre diciembre de 2009 y diciembre de 2011. Esta información constituye la base para desarrollar el pronóstico semanal de demanda. Antes de entrenar los modelos, será necesario depurar las transacciones y verificar la integridad de las semanas para evitar distorsiones en las predicciones.

| Indicador | Valor obtenido | Significado |
|---|---|---|
| Fuente de datos | ZIP oficial de UCI | Se utilizó una fuente oficial del conjunto de datos. |
| API `ucimlrepo` | `DatasetNotFoundError` | La descarga mediante API falló, pero se resolvió utilizando el archivo ZIP. |
| Hojas integradas | 2 | Se combinaron los periodos `Year 2009-2010` y `Year 2010-2011`. |
| Número de registros | 1,067,371 | El conjunto contiene un volumen amplio de transacciones comerciales. |
| Número de columnas | 8 | Incluye información de facturas, productos, cantidades, precios, fechas, clientes y países. |
| Fecha inicial | 01/12/2009 | Inicio del periodo observado. |
| Fecha final | 09/12/2011 | Fin del periodo observado. |
| Variable temporal | `InvoiceDate` | Permite organizar las transacciones por semana. |
| Variable de cantidad | `Quantity` | Permite cuantificar las unidades vendidas después de la depuración. |

In [ ]:
# ══════════════════════════════════════════════════════════════
# ESTANDARIZACIÓN DE LOS NOMBRES DE LAS COLUMNAS
# ══════════════════════════════════════════════════════════════

# Identificación de columnas equivalentes

def find_col(columns, candidates):
    ## Normalizar los nombres originales
    normalized = {
        str(c).lower().replace(" ", "").replace("_", ""): c
        for c in columns
    }  ### Relacionar los nombres normalizados con los nombres originales

    ## Buscar coincidencias entre los nombres candidatos
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")  ### Normalizar el nombre candidato

        if key in normalized:  ### Comprobar si existe una coincidencia
            return normalized[key]  ### Devolver el nombre original encontrado

    return None  ### Indicar que no se encontró coincidencia


# Relacionar los nombres originales con los nombres estandarizados

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),  ### Identificar el número de factura
    "StockCode": find_col(df.columns, ["StockCode"]),  ### Identificar el código del producto
    "Description": find_col(df.columns, ["Description"]),  ### Identificar la descripción del producto
    "Quantity": find_col(df.columns, ["Quantity"]),  ### Identificar la cantidad registrada
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),  ### Identificar la fecha de la transacción
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),  ### Identificar el precio unitario
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),  ### Identificar el identificador del cliente
    "Country": find_col(df.columns, ["Country"]),  ### Identificar el país de la transacción
}


## Verificar las columnas obligatorias

faltan = [
    k for k, v in mapping.items()
    if v is None and k not in ("Description", "CustomerID")
]  ### Obtener las columnas obligatorias que no se identificaron

if faltan:  ### Comprobar si existen columnas obligatorias ausentes
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")  ### Mostrar el error y detener la ejecución


# Renombrar las columnas identificadas

df = df.rename(columns={
    v: k for k, v in mapping.items() if v is not None
})  ### Aplicar los nombres estandarizados sin modificar los valores


## Mostrar la correspondencia de los nombres

print(mapping)  ### Mostrar las columnas originales identificadas
print("\nColumnas finales:", df.columns.tolist())  ### Verificar los nombres finales del DataFrame

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}

Columnas finales: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


### **Estandarización correcta de las columnas del conjunto de datos**

El código identificó correctamente las ocho columnas del conjunto de datos y confirmó que sus nombres coinciden con los nombres estandarizados definidos en `mapping`. La validación se completó sin errores y no fue necesario renombrar ninguna columna. Esto permite continuar con la limpieza y preparación de los datos para construir el pronóstico semanal de demanda.

| Indicador | Valor obtenido | Significado |
|---|---|---|
| Columnas identificadas | 8 | Se reconocieron todas las columnas esperadas. |
| Columnas obligatorias faltantes | 0 | No se detectaron campos esenciales ausentes. |
| Columnas renombradas | 0 | Los nombres originales ya coincidían con los estandarizados. |
| Validación | Correcta | El código finalizó sin generar errores. |
| Estado del DataFrame | Preparado para continuar | Se puede avanzar a la revisión de calidad y limpieza de los datos. |

# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONTROL DE CALIDAD Y DEPURACIÓN DE LOS DATOS
# ══════════════════════════════════════════════════════════════

# Conversión de tipos de datos

## Convertir las variables a sus tipos correspondientes

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")  ### Convertir las fechas y transformar los valores inválidos en nulos
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")  ### Convertir las cantidades a valores numéricos
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")  ### Convertir los precios unitarios a valores numéricos


# Depuración de los códigos de producto

## Identificar y eliminar códigos de producto ausentes

print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))  ### Contar los códigos nulos antes de eliminarlos
df = df[df["StockCode"].notna()].copy()  ### Conservar únicamente los registros con código de producto

df["StockCode"] = df["StockCode"].astype(str).str.strip()  ### Convertir los códigos a texto y eliminar espacios externos
df = df[df["StockCode"] != ""].copy()  ### Descartar los códigos vacíos después de la conversión


# Diagnóstico de la calidad de los registros

## Identificar las facturas de cancelación

cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")  ### Detectar las facturas cuyo identificador comienza con C


## Revisar el periodo temporal y los registros problemáticos

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())  ### Mostrar la fecha inicial y el día de la semana
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())  ### Mostrar la fecha final y el día de la semana
print("Cancelaciones:", int(cancel.sum()))  ### Contar las facturas identificadas como cancelaciones
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))  ### Contar los registros con cantidades nulas o negativas
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))  ### Contar los registros con precios nulos o negativos
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))  ### Contar los registros sin una fecha válida


# Filtrado de transacciones válidas

## Aplicar los criterios de limpieza

clean = df.loc[
    (~cancel) &  ### Excluir las facturas identificadas como cancelaciones
    df["InvoiceDate"].notna() &  ### Conservar registros con fecha válida
    (df["Quantity"] > 0) &  ### Conservar cantidades positivas
    (df["UnitPrice"] > 0) &  ### Conservar precios unitarios positivos
    df["Country"].notna()  ### Exigir que el país esté informado
].copy()  ### Crear un DataFrame independiente con las transacciones válidas


# Selección del mercado objetivo

## Filtrar las transacciones del Reino Unido

uk = clean[
    clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")
].copy()  ### Conservar únicamente las transacciones del Reino Unido


# Resumen de la depuración

## Mostrar el volumen de datos disponibles para el análisis

print("Filas válidas UK:", len(uk))  ### Mostrar el número de transacciones válidas del Reino Unido
print("SKUs UK:", uk["StockCode"].nunique())  ### Contar los productos distintos disponibles para el pronóstico

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501
SKUs UK: 4905


### **Depuración de transacciones y selección del mercado objetivo**

La depuración permitió identificar y excluir las transacciones que no cumplen los criterios de calidad establecidos, como facturas de cancelación, cantidades no positivas, precios unitarios no positivos y fechas inválidas. Posteriormente, se seleccionaron las operaciones correspondientes al Reino Unido, obteniendo **958,501 transacciones válidas y 4,905 productos distintos (SKUs)**. Este subconjunto servirá como base para analizar la demanda semanal y seleccionar el producto con mejores condiciones para desarrollar el pronóstico.

| Indicador | Valor obtenido | Significado |
|---|---:|---|
| Códigos de producto nulos | 0 | No se encontraron códigos de producto ausentes en el diagnóstico. |
| Fecha inicial | 01/12/2009, martes | Inicio del periodo registrado. |
| Fecha final | 09/12/2011, viernes | Fin del periodo registrado. |
| Facturas de cancelación | 19,494 | Registros identificados mediante facturas cuyo código comienza con `C`. |
| Cantidades menores o iguales a cero | 22,950 | Registros que requieren exclusión del análisis de demanda positiva. |
| Precios unitarios menores o iguales a cero | 6,207 | Registros excluidos por no cumplir el criterio de precio positivo. |
| Fechas nulas | 0 | Todas las fechas superaron la verificación de valores ausentes. |
| Transacciones válidas del Reino Unido | 958,501 | Registros que cumplen los filtros aplicados para el mercado objetivo. |
| Productos distintos del Reino Unido | 4,905 | Cantidad de SKUs disponibles para evaluar y seleccionar el producto que se pronosticará. |

### Pregunta 1
**¿Por qué no sería correcto sumar directamente `Quantity` sin eliminar cancelaciones y cantidades no positivas?**

No sería correcto sumar directamente Quantity porque el dataset contiene cancelaciones, devoluciones y cantidades no positivas. Estos registros pueden reducir artificialmente las ventas o distorsionar la demanda real del producto. Para construir un pronóstico semanal confiable, primero se deben conservar las transacciones válidas según los criterios de limpieza establecidos.

# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. SEMANAS COMPLETAS Y PARTICIÓN TEMPORAL
# ══════════════════════════════════════════════════════════════

# Identificación del lunes de cada semana

def lunes(fechas):
    fechas = pd.to_datetime(fechas)  ### Convertir las fechas al formato datetime
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")  ### Obtener el lunes de cada semana


## Determinación de las semanas completas

uk["week"] = lunes(uk["InvoiceDate"])  ### Asignar el lunes correspondiente a cada transacción
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()  ### Obtener la primera y última fecha del archivo
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())  ### Calcular el lunes de la primera semana
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())  ### Calcular el lunes de la última semana
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)  ### Determinar el inicio de la primera semana completa
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)  ### Determinar el inicio de la última semana completa


## Construcción del calendario y partición temporal

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")  ### Generar las fechas de inicio de las semanas completas
TEST_H = 8  ### Definir las últimas ocho semanas como test final
semanas_test = semanas[-TEST_H:]  ### Seleccionar las ocho semanas finales para la evaluación
semanas_train = semanas[:-TEST_H]  ### Reservar las semanas anteriores para entrenamiento y validación


## Identificación de semanas sin registros

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)  ### Contar las transacciones registradas en cada semana
semanas_sin_registro = registros[registros == 0].index  ### Identificar las semanas sin transacciones registradas


## Presentación de la partición temporal

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())  ### Mostrar los lunes de las semanas inicial y final del archivo
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())  ### Mostrar la cantidad y el periodo de semanas completas
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())  ### Mostrar la cantidad y el periodo de entrenamiento
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())  ### Mostrar la cantidad y el periodo del test final
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])  ### Mostrar las semanas sin registros, sin asumir que la tienda estuvo cerrada

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


### **La partición temporal permite evaluar el pronóstico sin utilizar información futura**

Se identificaron **104 semanas completas** entre el 7 de diciembre de 2009 y el 28 de noviembre de 2011. De ellas, 96 semanas se reservan para entrenamiento y validación, mientras que las últimas 8 semanas, del 10 de octubre al 28 de noviembre de 2011, se destinan al test final. Se detectaron dos semanas sin transacciones registradas en los datos filtrados: el 28 de diciembre de 2009 y el 27 de diciembre de 2010. La ausencia de registros no confirma que la tienda haya estado cerrada.

| Indicador | Valor | Significado |
|---|---:|---|
| Semanas completas | 104 | Periodo disponible para el análisis |
| Entrenamiento y validación | 96 semanas | Periodo para desarrollar y seleccionar el modelo |
| Test final | 8 semanas | Periodo reservado para la evaluación final |
| Semanas sin registros | 2 | Semanas sin transacciones en los datos filtrados |
| Primera semana completa | 07/12/2009 | Inicio de la serie semanal |
| Última semana completa | 28/11/2011 | Fin de la serie semanal |

### Pregunta 2
**¿Por qué una partición aleatoria con `train_test_split` sería incorrecta en este problema?**

Porque train_test_split divide los datos aleatoriamente, mezclando semanas pasadas y futuras entre entrenamiento y test. Esto puede provocar fuga de información temporal y generar una evaluación poco realista. La partición temporal respeta el orden cronológico y permite evaluar el modelo con las últimas ocho semanas, simulando un pronóstico de demanda real.

# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. SELECCIÓN REPRODUCIBLE DEL PRODUCTO
# ══════════════════════════════════════════════════════════════

# Filtrado de las transacciones del periodo de entrenamiento

uk_train = uk[uk["week"].isin(semanas_train)]  ### Conservar únicamente las transacciones de las semanas de entrenamiento
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]  ### Identificar las semanas de entrenamiento con registros en la tienda


## Cálculo de la cobertura y las unidades por producto

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()  ### Sumar las unidades vendidas por producto y semana
             .groupby("StockCode")  ### Agrupar los registros por producto
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),  ### Calcular las semanas con ventas y las unidades totales
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))  ### Identificar la primera y última semana con ventas
             .reset_index())  ### Convertir el índice en una columna

cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)  ### Calcular la proporción de semanas con ventas


## Aplicación de los criterios de elegibilidad

elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)  ### Exigir cobertura mínima del 90 %
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)  ### Ordenar por unidades y semanas con ventas, de mayor a menor


## Verificación de productos elegibles

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")  ### Detener el proceso si ningún producto cumple los criterios

print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)  ### Mostrar la cantidad de productos elegibles y verificar si hay al menos dos


## Selección del producto con mayor volumen de ventas

SKU = elegibles.loc[0, "StockCode"]  ### Seleccionar el producto con más unidades entre los elegibles
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]  ### Obtener la descripción más frecuente del producto

print("SKU seleccionado:", SKU, "·", desc)  ### Mostrar el código y la descripción del producto seleccionado
display(elegibles.head(5))  ### Mostrar los cinco primeros productos elegibles

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


### **Se identificaron 389 productos elegibles y se seleccionó el de mayor volumen de ventas**

La selección se realizó exclusivamente con las semanas de entrenamiento para evitar utilizar información del test final. Se identificaron 389 productos que cumplieron los criterios establecidos y se seleccionó el SKU **84077, WORLD WAR 2 GLIDERS ASSTD DESIGNS**, por registrar la mayor cantidad de unidades vendidas, con 88,273 unidades en 93 semanas. Su cobertura fue de 98.94 %, superior al mínimo requerido del 90 %. Además, se cuenta con suficientes productos elegibles para desarrollar el Reto 3. Aunque otros productos presentaron ventas en más semanas, el SKU 84077 fue seleccionado por tener el mayor volumen total de unidades entre los productos elegibles.

| Indicador | Valor | Significado |
|---|---:|---|
| Productos elegibles | 389 | Productos que cumplieron los criterios de selección |
| SKU seleccionado | 84077 | Producto elegido para el pronóstico |
| Descripción | WORLD WAR 2 GLIDERS ASSTD DESIGNS | Nombre del producto seleccionado |
| Unidades vendidas | 88,273 | Mayor volumen de ventas entre los productos elegibles |
| Semanas con ventas | 93 | Semanas con ventas registradas durante el entrenamiento |
| Primera semana con ventas | 07/12/2009 | Inicio de las ventas registradas del producto |
| Última semana con ventas | 03/10/2011 | Última semana con ventas dentro del entrenamiento |
| Cobertura | 98.94 % | Proporción de semanas con ventas respecto a las semanas con registros en la tienda |
| Cobertura mínima requerida | 90 % | Criterio cumplido por el producto seleccionado |
| Suficientes productos para el Reto 3 | Sí | Hay al menos dos productos elegibles |

# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. SERIE SEMANAL DE VENTAS REGISTRADAS
# ══════════════════════════════════════════════════════════════

# Construcción de la serie semanal del producto seleccionado

serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()  ### Sumar las unidades vendidas del SKU por semana
         .reindex(semanas, fill_value=0).astype(float))  ### Completar las semanas sin ventas registradas con cero y convertir a float
serie.name = "unidades"  ### Asignar un nombre descriptivo a la serie


## Separación de entrenamiento y test final

train = serie.loc[semanas_train]  ### Extraer las ventas registradas del periodo de entrenamiento
test = serie.loc[semanas_test]  ### Extraer las ventas registradas del periodo de test final


## Identificación de semanas sin ventas registradas

ceros = serie[serie == 0].index  ### Identificar las semanas con cero unidades registradas

print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))  ### Mostrar la extensión de la serie y la cantidad de semanas con cero ventas
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])  ### Identificar los ceros que coinciden con semanas sin actividad registrada en la tienda
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])  ### Identificar los ceros cuando sí hubo registros de otras ventas en la tienda
print("Unidades en entrenamiento:", int(train.sum()))  ### Mostrar el total de unidades registradas durante el entrenamiento

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


### **La serie semanal permite analizar las ventas registradas y detectar semanas sin actividad del producto**

La serie comprende 104 semanas completas. Durante las 96 semanas de entrenamiento se registraron 88,273 unidades del SKU seleccionado. Se identificaron tres semanas sin ventas registradas del producto: dos coincidieron con semanas sin registros en toda la tienda, correspondientes al 28 de diciembre de 2009 y al 27 de diciembre de 2010. La tercera, el 17 de enero de 2011, presentó registros de otras ventas, pero ninguna del producto seleccionado. Estos ceros representan ausencia de ventas registradas y no confirman demanda nula ni disponibilidad de inventario.

| Indicador | Valor | Significado |
|---|---:|---|
| Semanas de la serie | 104 | Total de semanas completas analizadas |
| Unidades en entrenamiento | 88,273 | Total de unidades vendidas durante el entrenamiento |
| Semanas sin ventas del SKU | 3 | Semanas con cero unidades registradas |
| Semanas sin registros en la tienda | 2 | No se registraron transacciones en toda la tienda |
| Semanas con registros en la tienda, pero sin ventas del SKU | 1 | Hubo actividad comercial de otros productos |

# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. EXPLORACIÓN DE LAS VENTAS REGISTRADAS
# ══════════════════════════════════════════════════════════════

# Análisis de la evolución semanal y las medias móviles

explora = pd.DataFrame({"ventas": train})  ### Crear el DataFrame con las ventas del periodo de entrenamiento
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()  ### Calcular la media móvil de las últimas cuatro semanas
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()  ### Calcular la media móvil de las últimas ocho semanas

fig = go.Figure()  ### Crear la figura para comparar las ventas y las medias móviles

for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))  ### Añadir cada serie al gráfico

fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")  ### Configurar el título y los ejes del gráfico
fig.show()  ### Mostrar la evolución semanal de las ventas


## Análisis del perfil promedio por semana del año

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))  ### Añadir el número de semana del año a cada observación
          .groupby("semana_anio")["ventas"].mean().reset_index())  ### Calcular las ventas promedio para cada semana del año

fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,  ### Crear un gráfico de líneas con marcadores
              title="Perfil promedio por semana del año (entrenamiento)",  ### Definir el título del gráfico
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})  ### Nombrar los ejes del gráfico
fig.show()  ### Mostrar el perfil promedio semanal


## Resumen estadístico anual de las ventas

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])  ### Calcular cantidad de semanas, media, mediana y máximo por año
display(anual.round(1))  ### Mostrar las estadísticas redondeadas a un decimal

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


### **Las ventas presentan alta variabilidad, picos estacionales y cambios en el nivel semanal**

La exploración de las 96 semanas de entrenamiento muestra una elevada variabilidad en las ventas del SKU 84077, con picos pronunciados entre octubre y diciembre de 2010. La media móvil de cuatro semanas reacciona más rápidamente a estos cambios, mientras que la de ocho semanas presenta una evolución más suavizada. En el perfil promedio por semana del año destacan valores elevados hacia el final del año, especialmente en las semanas 44 y 48. Asimismo, la media semanal fue de 1,034.4 unidades en 2010 y de 812.8 en 2011. Estos patrones sugieren variaciones temporales y posibles efectos estacionales, aunque la exploración por sí sola no confirma una estacionalidad estable.

| Indicador | Valor | Significado |
|---|---:|---|
| Media semanal en 2009 | 493.0 unidades | Solo considera 4 semanas de entrenamiento |
| Media semanal en 2010 | 1,034.4 unidades | Mayor nivel promedio anual observado |
| Media semanal en 2011 | 812.8 unidades | Menor promedio que en 2010 |
| Máximo semanal en 2010 | 7,063 unidades | Pico pronunciado de ventas registradas |
| Máximo semanal en 2011 | 3,216 unidades | Pico inferior al máximo de 2010 |
| Medias móviles | 4 y 8 semanas | Herramientas exploratorias para observar cambios de nivel |

### Pregunta 3
**Identifique evidencia de tendencia, estacionalidad o quiebres en el entrenamiento. Diferencie lo que observa de lo que todavía necesita comprobar, y explique qué representa un cero en la serie.**

Se observa alta variabilidad en las ventas, con picos pronunciados entre octubre y diciembre de 2010. El perfil por semana del año sugiere una posible estacionalidad, mientras que la reducción de la media semanal de 1,034.4 unidades en 2010 a 812.8 en 2011 podría indicar un cambio de nivel. Sin embargo, se necesita validación temporal para confirmar si existe estacionalidad recurrente, tendencia sostenida o un quiebre estructural.

Un cero representa una semana sin ventas registradas del producto. Puede corresponder a ausencia de registros en toda la tienda o a una semana con ventas de otros productos, pero ninguna del SKU analizado. No demuestra demanda nula ni disponibilidad de inventario.

# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [ ]:

# ══════════════════════════════════════════════════════════════
# 8. MODELOS, CANTIDAD OPERATIVA, COSTO Y REGLA DE SELECCIÓN
# ══════════════════════════════════════════════════════════════

# Configuración de los modelos y los costos

SEASON = 4  ### Definir el periodo estacional de cuatro semanas
OVER_COST, SHORT_COST = 0.40, 1.50  ### Establecer los costos por unidad excedente y faltante en libras
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]  # Ordenar los modelos de menor a mayor complejidad
TOLERANCIA = 0.01  ### Permitir que el costo promedio esté hasta un 1 % por encima del mínimo
registro_ajustes = []  ### Crear una lista para registrar los ajustes y las advertencias del modelo


## Ajuste del modelo Exponential Smoothing

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:  ### Verificar si hay suficientes observaciones para utilizar el componente estacional
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")  ### Configurar nivel, tendencia y estacionalidad aditivos con periodo de cuatro semanas
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"  ### Describir la estructura completa del modelo
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")  ### Utilizar nivel y tendencia cuando no hay suficientes observaciones
        especificacion = "simplificado: nivel + tendencia"  ### Registrar la estructura simplificada del modelo
    with warnings.catch_warnings(record=True) as avisos:  ### Capturar las advertencias generadas durante el ajuste
        warnings.simplefilter("always")  ### Registrar todas las categorías de advertencias
        ajuste = modelo.fit(optimized=True)  ### Estimar los parámetros del modelo mediante optimización
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),  ### Registrar la etapa, el ajuste y el tamaño del entrenamiento
                             "especificacion": especificacion, "advertencias": len(avisos),  ### Guardar la especificación y la cantidad de advertencias
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),  ### Registrar los tipos de advertencias detectadas
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})  ### Guardar los mensajes de advertencia resumidos
    return ajuste  ### Devolver el modelo ajustado


## Generación de pronósticos y conversión a cantidades operativas

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)  ### Repetir la última observación para todas las semanas del horizonte
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)  ### Repetir el último ciclo observado de cuatro semanas
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)  ### Ajustar Exponential Smoothing y generar el pronóstico futuro
    else:
        raise ValueError(modelo)  ### Generar un error si se solicita un modelo no reconocido
    return np.maximum(np.round(f), 0)   # Redondear los pronósticos y evitar cantidades negativas para obtener valores operativos


## Evaluación del error y los costos operativos

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)  ### Convertir las ventas reales y las cantidades pronosticadas en arreglos numéricos
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)  ### Calcular las unidades excedentes y las unidades faltantes por semana
    return {"MAE": mean_absolute_error(real, cantidad),  ### Calcular el error absoluto medio entre las ventas reales y pronosticadas
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,  ### Calcular la raíz del error cuadrático medio para penalizar más los errores grandes
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # Calcular el error porcentual absoluto ponderado; no definido si las ventas reales suman cero
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),  ### Sumar las unidades excedentes y faltantes de todo el periodo evaluado
            "costo": (exceso * over + faltante * short).sum()}  ### Calcular el costo total ponderando los excedentes y faltantes con sus respectivos costos unitarios


## Resumen de las métricas entre particiones de validación

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)  ### Agrupar los resultados según el modelo o la política indicada
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),  ### Calcular el costo promedio y su desviación estándar entre folds
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),  ### Resumir el error absoluto medio, su variabilidad y el error cuadrático medio
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),  ### Calcular el WAPE promedio y sumar las unidades excedentes
                faltantes=("unidades_faltantes", "sum"))  ### Sumar las unidades faltantes
           .reindex(orden))  ### Ordenar los resultados según la lista proporcionada
    t["complejidad"] = range(len(orden))  ### Asignar un nivel de complejidad según el orden definido
    return t  ### Devolver la tabla resumen con las métricas calculadas


## Selección mediante costo, MAE, variabilidad y complejidad

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()  ### Identificar el costo promedio mínimo entre las alternativas
    limite = minimo * (1 + tol)          # Establecer el costo máximo permitido dentro de la tolerancia; si el mínimo es cero, el límite también será cero
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]  ### Filtrar los modelos cuyo costo promedio se encuentra dentro del límite permitido
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]  ### Elegir el candidato con menor MAE, menor variabilidad del costo y menor complejidad, en ese orden


## Confirmación de la configuración

print("Funciones listas.")  ### Confirmar que las funciones y la configuración quedaron definidas


Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. BACKTESTING ROLLING-ORIGIN (SOLO CON EL ENTRENAMIENTO)
# ══════════════════════════════════════════════════════════════

# Configuración del backtesting

BT_H, MIN_TRAIN = 8, 52  ### Definir el horizonte de validación y el mínimo de semanas de entrenamiento
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]  ### Generar los orígenes de validación en orden cronológico


## Función de validación temporal

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}  ### Inicializar las métricas y los pronósticos de cada fold
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]  ### Separar el entrenamiento acumulado y las ocho semanas de validación
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"  ### Comprobar que el fold respete el orden temporal y no invada el test
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")  ### Generar las cantidades operativas pronosticadas
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})  ### Registrar las métricas de evaluación
    return filas, cantidades  ### Devolver las métricas y las cantidades pronosticadas


## Ejecución del backtesting y registro de los folds

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")  ### Evaluar todos los modelos en cada fold temporal
bt = pd.DataFrame(filas_bt)  ### Convertir las métricas en una tabla

folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,  ### Registrar el periodo y tamaño del entrenamiento de cada fold
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}  ### Registrar las ocho semanas de validación
                      for k, o in enumerate(origenes, start=1)])  ### Crear una fila por cada origen temporal

display(folds)  ### Mostrar los periodos de entrenamiento y validación
display(bt.round(2))  ### Mostrar las métricas de cada modelo y fold con dos decimales

print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")  ### Confirmar que todos los folds terminan antes del test final

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


### **El backtesting permite comparar los modelos bajo diferentes periodos de validación**

Se evaluaron tres modelos mediante cinco folds temporales, cada uno con ocho semanas de validación y un entrenamiento creciente de 56 a 88 semanas. El modelo **ExpSmoothing obtuvo el menor costo en cuatro de los cinco folds y el menor costo acumulado promedio por fold**, con £4,324.20, aproximadamente. Sin embargo, SeasonalNaive presentó menor MAE en los folds 1 y 2, mientras que Naive obtuvo menor MAE en los folds 3, 4 y 5. Esto evidencia que el desempeño depende del periodo evaluado y que la selección final debe considerar el costo operativo de los excedentes y faltantes.

| Indicador | Valor o hallazgo | Significado |
|---|---:|---|
| Folds evaluados | 5 | Se compararon cinco periodos de validación |
| Horizonte por fold | 8 semanas | Cada modelo pronostica ocho semanas futuras |
| Entrenamiento | 56 a 88 semanas | La cantidad de datos aumenta progresivamente |
| Mejor costo por fold | ExpSmoothing en 4 de 5 folds | Presentó menor costo operativo en la mayoría de los periodos |
| Costo promedio por fold de ExpSmoothing | £4,324.20 | Menor costo promedio entre los tres modelos |
| Menor MAE por fold | SeasonalNaive en folds 1 y 2; Naive en folds 3, 4 y 5 | El menor error absoluto depende del periodo |
| Separación temporal | Verificada | Ningún fold invade el test final, que empieza el 10/10/2011 |

In [ ]:
# Resumen de métricas y selección del modelo

resumen_val = resumir(filas_bt, "modelo", MODELOS)  ### Resumir las métricas de desempeño de cada modelo en los folds de validación
SELECCIONADO = seleccionar(resumen_val)  ### Seleccionar el modelo según la regla establecida de costo, MAE, variabilidad y complejidad
display(resumen_val.round(2))  ### Mostrar las métricas resumidas con dos decimales
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)  ### Mostrar el modelo seleccionado antes de evaluar el conjunto de prueba


## Comparación gráfica de los costos por fold

fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",  ### Crear un diagrama de caja para comparar los costos de los modelos entre folds
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})  ### Definir las etiquetas de los ejes del gráfico
fig.show()  ### Mostrar la distribución y los valores individuales de los costos por modelo


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


### **ExpSmoothing presenta el menor costo operativo promedio**

El modelo ExpSmoothing fue seleccionado en la validación porque obtuvo el menor costo operativo promedio, con £4,324.10 por fold, superando a SeasonalNaive (£4,950.34) y Naive (£6,239.22). Sin embargo, presentó un MAE promedio más alto (927.3 unidades), lo que indica que la selección prioriza el costo operativo definido para los excedentes y faltantes, no únicamente la precisión del pronóstico. El diagrama de cajas muestra que los costos varían entre los cinco folds, por lo que el desempeño depende del periodo evaluado. El modelo queda seleccionado antes de consultar el conjunto de test, manteniendo la separación temporal de la evaluación.

| Indicador | Naive | SeasonalNaive | ExpSmoothing |
|---|---:|---:|---:|
| Costo promedio (£) | 6,239.22 | 4,950.34 | **4,324.10** |
| Desviación estándar del costo (£) | 4,296.41 | 2,065.83 | 2,335.11 |
| MAE promedio (unidades) | **620.2** | 666.5 | 927.3 |
| RMSE promedio | 819.13 | 830.61 | 1,063.63 |
| WAPE promedio | 0.75 | 0.83 | 1.29 |
| Unidades excedentes | 5,469 | 13,853 | 30,925 |
| Unidades faltantes | 19,339 | 12,807 | 6,167 |

La selección es coherente con el criterio establecido: los faltantes cuestan £1.50 por unidad, mientras que los excedentes cuestan £0.40. Por ello, ExpSmoothing logra el menor costo al reducir los faltantes, aunque genera más excedentes y presenta mayores errores de pronóstico.

### Pregunta 4
**¿Qué modelo se seleccionó en validación y con qué regla? ¿Coinciden el ranking por costo y el ranking por MAE o RMSE? ¿Por qué el ranking por WAPE no aporta un criterio adicional al MAE?**

Se seleccionó **ExpSmoothing** mediante una regla que prioriza el menor costo operativo promedio. Se admitieron modelos con un costo de hasta 1 % por encima del mínimo y, entre los candidatos, se aplicaron como desempate el menor MAE promedio, la menor desviación estándar del costo y la menor complejidad, en ese orden.

Los rankings por costo y por MAE o RMSE no coinciden. ExpSmoothing obtuvo el menor costo promedio (£4,324.10), pero Naive presentó el menor MAE (620.2) y RMSE (819.13). Esto ocurre porque el costo asigna penalizaciones diferentes a los errores: £0.40 por unidad excedente y £1.50 por unidad faltante. Por ello, un modelo con mayor error puede resultar económicamente más conveniente si reduce los faltantes.

El WAPE no aporta un criterio adicional al MAE porque ambos se calculan a partir del error absoluto. Para un mismo conjunto de evaluación, el WAPE es el MAE multiplicado por el número de observaciones y dividido entre la suma de las ventas reales. Por tanto, ambos producen el mismo ranking de modelos cuando se evalúan sobre las mismas semanas.

### Pregunta 5
**¿Qué tan estable fue cada modelo entre folds? Use la desviación del costo y del MAE.**

El modelo **SeasonalNaive fue el más estable en términos de costo**, con una desviación estándar de £2,065.83, seguido de ExpSmoothing (£2,335.11) y Naive (£4,296.41). Esto indica que Naive presentó la mayor variabilidad de costos entre los periodos de validación. Respecto al MAE, Naive también fue el más estable, con una desviación estándar de 327.68 unidades, seguido de SeasonalNaive (370.02) y ExpSmoothing (420.72). Por tanto, ExpSmoothing logró el menor costo promedio, pero mostró mayor variabilidad del error absoluto, mientras que SeasonalNaive presentó el comportamiento más consistente en costos.

| Modelo | Desviación del costo (£) | Desviación del MAE (unidades) |
|---|---:|---:|
| Naive | 4,296.41 | **327.68** |
| SeasonalNaive | **2,065.83** | 370.02 |
| ExpSmoothing | 2,335.11 | 420.72 |

# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [ ]:

# ══════════════════════════════════════════════════════════════
# 10. POLÍTICA DE COLCHÓN CALIBRADA SIN INFORMACIÓN FUTURA
# ══════════════════════════════════════════════════════════════

# Configuración de los multiplicadores y bloques de calibración

MULTIPLICADORES = [0, 0.5, 1.0, 1.5]  ### Definir los multiplicadores candidatos para calcular las unidades adicionales de inventario
BLOQUES_CAL = 2  ### Establecer dos bloques históricos de ocho semanas para calcular el MAE previo


## Cálculo del MAE previo

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []  ### Inicializar la lista de errores absolutos de los bloques históricos
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H  ### Determinar el inicio de cada bloque histórico de ocho semanas
        if corte < 2 * SEASON + 8:
            return None  ### Indicar que no hay suficientes datos históricos para entrenar el modelo
        real = y.iloc[corte:corte + BT_H].values  ### Extraer las ventas reales del bloque que se evaluará
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))  ### Pronosticar usando solo los datos anteriores al bloque y guardar los errores absolutos
    return float(np.mean(np.concatenate(errores)))  ### Calcular el MAE conjunto de todos los bloques históricos


## Cálculo de las unidades adicionales de inventario

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))  ### Calcular el colchón como el multiplicador por el MAE, redondeado hacia arriba; devolver cero si no hay historial suficiente o el multiplicador es cero


## Evaluación comparativa de las políticas de colchón

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []  ### Inicializar la lista que almacenará los resultados de cada política
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]  ### Separar el entrenamiento disponible y el bloque de validación de ocho semanas
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)  ### Calcular el MAE histórico del modelo seleccionado usando únicamente información previa al origen
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)  ### Calcular el colchón correspondiente al multiplicador actual
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,  ### Evaluar el pronóstico ajustado y registrar el fold y el multiplicador
                          "mae_previo": mae_k, "colchon": c,  ### Guardar el MAE previo y las unidades adicionales aplicadas
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})  ### Registrar las fechas para verificar que la calibración preceda a la validación
    return pd.DataFrame(filas)  ### Devolver los resultados de las políticas en una tabla


## Selección de la política y preparación del test

pol_folds = comparar_politicas()  ### Evaluar cada multiplicador en todos los folds de validación
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"  ### Comprobar que la calibración no utilice información futura
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)  ### Resumir los costos y errores de cada política
MULT = seleccionar(resumen_pol)  ### Seleccionar el multiplicador aplicando la regla única de selección

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))  ### Mostrar las fechas y el MAE previo para el multiplicador de 1.0
display(resumen_pol.round(2))  ### Mostrar las métricas comparativas de las políticas con dos decimales

MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")  # Calcular el MAE previo al test usando exclusivamente el entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)  ### Calcular las unidades adicionales que se aplicarán durante el test
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"  ### Construir la descripción de la política seleccionada
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)  ### Mostrar la política elegida antes de evaluar el test
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")  ### Mostrar el colchón y el MAE histórico utilizado para calcularlo


,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


### **La política sin colchón minimiza el costo operativo promedio**

Los resultados muestran que el multiplicador **0.0** obtiene el menor costo promedio (£4,324.10), por lo que se selecciona la política **ExpSmoothing sin unidades adicionales de inventario**. Aunque los multiplicadores mayores reducen los faltantes, incrementan los excedentes y el costo total. La política elegida se fija antes de evaluar el conjunto de prueba, utilizando únicamente información del entrenamiento.

| Indicador | Valor | Significado |
|---|---:|---|
| Multiplicador seleccionado | 0.0 | No se agregan unidades adicionales al pronóstico. |
| Costo promedio | £4,324.10 | Menor costo promedio entre las políticas evaluadas. |
| Desviación estándar del costo | £2,335.11 | Variabilidad del costo entre los folds. |
| Excedentes | 30,925 unidades | Unidades pronosticadas por encima de la demanda. |
| Faltantes | 6,167 unidades | Unidades de demanda que no se cubrirían. |
| MAE previo para el test | 618.8 | Error histórico usado para calcular el colchón. |
| Colchón para el test | 0 unidades/semana | Resultado de aplicar el multiplicador seleccionado. |

Los valores de calibración (`mae_previo`) disminuyen de 1,400.7 a 821.3 en los cinco folds, aunque presentan fluctuaciones. Esto refleja cambios en el error reciente del modelo. Para el test, el MAE previo es 618.8; sin embargo, como el multiplicador seleccionado es cero, no se añaden unidades al pronóstico.

**Importante:** esta decisión minimiza el costo bajo los supuestos establecidos (£0.40 por excedente y £1.50 por faltante). No significa que sea la política con menos faltantes, sino que ofrece el menor costo promedio según la función de costos definida.

### Pregunta 6
**¿Qué política de colchón quedó fijada antes del test y por qué? Explique por qué el colchón de cada fold se calcula solo con errores anteriores a su origen.**

La política fijada antes del test fue ExpSmoothing + 0 × MAE previo, es decir, no agregar unidades adicionales al pronóstico, porque obtuvo el menor costo promedio (£4,324.10) durante la validación. El colchón de cada fold se calcula únicamente con errores anteriores a su origen para evitar la fuga de información futura (data leakage) y simular una situación real en la que solo se dispone de datos históricos para tomar decisiones. Así, la evaluación es más confiable y representa mejor el desempeño que tendría la política en la práctica.

# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [ ]:

# ══════════════════════════════════════════════════════════════
# 11. EVALUACIÓN FINAL EN TEST (UNA SOLA VEZ)
# ══════════════════════════════════════════════════════════════

# Pronóstico de los tres modelos y aplicación de la política fijada
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}  # Pronostica las 8 semanas de test con cada modelo entrenado en las 96 semanas disponibles
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST  # Añade el colchón definido previamente al pronóstico del modelo seleccionado

# Evaluación de los modelos y de la política fijada
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]  # Calcula las métricas de cada modelo sin unidades adicionales
if MULT > 0:  # Comprueba si la política seleccionada agrega un colchón positivo
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})  # Incorpora la política con colchón solo si el multiplicador es mayor que cero
res_test = pd.DataFrame(filas).set_index("opcion")  # Organiza los resultados para comparar las alternativas evaluadas
display(res_test.round(2))  # Muestra las métricas redondeadas a dos decimales

# Comparación del costo frente al modelo Naive
fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"  # Identifica la fila que representa la política seleccionada
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]  # Extrae los costos de la política y del modelo de referencia
diferencia = costo_naive - costo_pol  # Calcula la diferencia: positiva indica ahorro y negativa indica sobrecosto

print("Política evaluada:", POLITICA)  # Informa la política evaluada en el conjunto de test
if SELECCIONADO == "Naive" and MULT == 0:  # Comprueba si la política seleccionada coincide con Naive sin colchón
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")  # Evita una comparación redundante
elif costo_naive == 0:  # Comprueba si el costo de referencia es cero
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")  # Informa la diferencia sin dividir entre cero
elif diferencia > 0:  # Identifica si la política cuesta menos que Naive
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")  # Muestra el ahorro absoluto y porcentual
else:  # Se ejecuta cuando la política iguala o supera el costo de Naive
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")  # Muestra el sobrecosto absoluto y porcentual

# Identificación de la mejor alternativa observada retrospectivamente
mejor_retro = res_test["costo"].idxmin()  # Identifica la alternativa con el menor costo en el test
if mejor_retro != fila_pol:  # Comprueba si otra alternativa superó a la política fijada
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")  # Reporta la diferencia sin modificar la selección previa

# Visualización de ventas reales y pronósticos
fig = go.Figure()  # Crea la figura del gráfico comparativo
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))  # Añade la serie de ventas reales del periodo de test
for m, q in cantidades_test.items():  # Recorre los pronósticos de los tres modelos
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))  # Añade al gráfico las cantidades pronosticadas por cada modelo
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")  # Configura el título y los ejes del gráfico
fig.show()  # Muestra la comparación visual del desempeño final


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


### **Naive obtiene el menor costo en el test final**

En las 8 semanas de prueba, el modelo Naive obtuvo el menor costo total (£7,087.90), con un MAE de 678.75. En cambio, la política fijada previamente, ExpSmoothing sin colchón, alcanzó un costo de £8,957.60, lo que representa un sobrecosto de £1,869.70 (26.4 %) frente a Naive. Aunque ExpSmoothing registró menos unidades faltantes, generó más excedentes, elevando el costo total. Por tanto, Naive habría sido la mejor alternativa retrospectivamente, pero se mantiene la selección original para respetar el procedimiento de evaluación.

| Indicador | Naive | ExpSmoothing |
|---|---:|---:|
| MAE | 678.75 | 776.62 |
| RMSE | 1,545.12 | 1,700.86 |
| WAPE | 0.48 | 0.54 |
| Unidades excedentes | 961 | 329 |
| Unidades faltantes | 7,087.9* | 8,957.6* |
| Costo total (£) | 7,087.90 | 8,957.60 |

\*En la tabla proporcionada, las columnas parecen haberse desplazado al copiar los resultados. Los valores 7,087.9 y 8,957.6 corresponden al costo total, no a las unidades faltantes. Se recomienda revisar la salida original de `res_test` para confirmar los faltantes y excedentes.

### Pregunta 7
**¿La política seleccionada cumplió en test? Compare su costo con los baselines e indique el ahorro o sobrecosto frente a Naïve. ¿El modelo con menor RMSE fue también el de menor costo?**

La política seleccionada, ExpSmoothing sin colchón, **no fue la de menor costo en el test**. Registró un costo de £8,957.60, mientras que Naive obtuvo £7,087.90 y SeasonalNaive £8,579.90. Frente a Naive, la política presentó un **sobrecosto de £1,869.70 (26.4 %)**. Por otro lado, el modelo con menor RMSE fue Naive, con 1,545.12, y también obtuvo el menor costo. Esto demuestra que, en este test, Naive tuvo el mejor desempeño tanto en precisión del pronóstico como en costo operativo.

| Indicador | Naive | SeasonalNaive | ExpSmoothing |
|---|---:|---:|---:|
| RMSE | 1,545.12 | 1,631.81 | 1,700.86 |
| Costo total | £7,087.90 | £8,579.90 | £8,957.60 |

### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [ ]:
# Identificación de la semana con mayor venta
semana_pico = test.idxmax()  # Obtiene la semana con el máximo de unidades vendidas en el test
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]  # Filtra las transacciones del SKU seleccionado durante la semana del pico

# Identificación del pedido más grande de la semana
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)  # Agrupa las unidades por factura y cliente, y ordena los pedidos de mayor a menor
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")  # Muestra la fecha y las unidades vendidas en la semana del pico

factura, cliente = pedidos.index[0]  # Recupera la factura y el cliente del pedido con más unidades
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")  # Normaliza el identificador del cliente y contempla valores ausentes
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")  # Muestra el tamaño del pedido y su proporción respecto a las ventas semanales
print("Mediana semanal del entrenamiento:", train.median(), "unidades")  # Compara el pico con la mediana de ventas semanales del entrenamiento

# Análisis retrospectivo de sensibilidad sin la semana del pico
sin_pico = test.index != semana_pico  # Crea una máscara para excluir únicamente la semana de mayor venta
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T  # Recalcula las métricas de cada modelo en las siete semanas restantes
if MULT > 0:  # Comprueba si la política seleccionada utiliza un colchón positivo
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])  # Evalúa la política fijada sin la semana del pico

print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")  # Aclara que el análisis es complementario y no sustituye la evaluación oficial
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))  # Presenta las métricas de sensibilidad redondeadas a dos decimales

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


### **El pico de ventas explica gran parte del error observado en el test**

La semana del 24 de octubre de 2011 registró 5,328 unidades, frente a una mediana semanal de 584.5 unidades en el entrenamiento. El pedido de la factura 573008 concentró 4,800 unidades, equivalentes al 90.1 % de las ventas de esa semana. Esto sugiere un pedido extraordinario puntual, aunque no permite confirmar por sí solo un cambio estructural. En el análisis retrospectivo sin esa semana, Naive obtuvo el menor MAE (155.86) y el menor costo (£579.40), superando a SeasonalNaive y ExpSmoothing. Esto indica que el pico influyó considerablemente en la evaluación, pero la evaluación principal debe conservarlo.

| Indicador sin la semana del pico | Naive | SeasonalNaive | ExpSmoothing |
|---|---:|---:|---:|
| MAE | 155.86 | 232.86 | 205.86 |
| Unidades excedentes | 961 | 641 | 329 |
| Unidades faltantes | 130 | 989 | 1,112 |
| Costo | £579.40 | £1,739.90 | £1,799.60 |

El resultado respalda que Naive fue la alternativa más económica en las siete semanas restantes. Sin embargo, este análisis es únicamente de sensibilidad retrospectiva y no sustituye la evaluación original de las ocho semanas.

### Pregunta 8
**¿Qué error fue más grave en el test: sobreestimar o subestimar? Use las unidades y los costos. ¿El pico del test indica un quiebre estructural o un pico puntual? ¿Qué muestra, y qué no, el análisis de sensibilidad sin esa semana?**

En el test, la subestimación generó un costo importante porque las unidades faltantes cuestan £1.50 cada una, frente a £0.40 por cada unidad excedente. Para la política seleccionada, ExpSmoothing, se registraron 329 unidades excedentes y 5,884 faltantes, por lo que predominó el costo asociado a la demanda no cubierta. Naive obtuvo un menor costo total de £7,087.90, frente a £8,957.60 de ExpSmoothing.

El pico del 24 de octubre de 2011 parece corresponder a un **pico puntual asociado a un pedido extraordinario**, ya que una sola factura concentró 4,800 de las 5,328 unidades vendidas esa semana (90.1 %), muy por encima de la mediana semanal del entrenamiento, de 584.5 unidades. Sin embargo, ocho semanas de test no permiten confirmar ni descartar definitivamente un quiebre estructural.

El análisis de sensibilidad sin la semana del pico muestra que Naive mantiene el menor costo (£579.40) y el menor MAE (155.86) entre las siete semanas restantes. Esto evidencia que el pico influyó considerablemente en la evaluación, pero **no demuestra que deba eliminarse ni reemplaza la evaluación principal**, que conserva las ocho semanas.

# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [ ]:

# ══════════════════════════════════════════════════════════════
# 12. REENTRENAMIENTO FINAL Y PROYECCIÓN ACADÉMICA
# ══════════════════════════════════════════════════════════════

# Configuración del horizonte de proyección
FINAL_H = 8  # Define las ocho semanas futuras que se desean pronosticar

# Reentrenamiento y generación de pronósticos
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")  # Reentrena el modelo seleccionado con las 104 semanas y pronostica las ocho semanas futuras
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")  # Genera el pronóstico de Naive como referencia comparativa

# Actualización de la magnitud del colchón
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")  # Calcula el MAE previo con los dos bloques más recientes de la historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)  # Actualiza las unidades adicionales aplicando el multiplicador seleccionado previamente

# Generación de las fechas futuras
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")  # Genera las fechas de los ocho lunes posteriores a la última semana disponible

# Construcción de la tabla de proyección académica
plan = pd.DataFrame({"Semana": semanas_futuras.date,  # Registra las fechas de las semanas proyectadas
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),  # Incluye las unidades pronosticadas por el modelo seleccionado
                     "Colchón (política fijada)": COLCHON_FINAL,  # Registra las unidades adicionales calculadas con la política fijada
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),  # Calcula el objetivo semanal sumando el pronóstico y el colchón
                     "Referencia Naïve": pron_naive.astype(int)})  # Añade las cantidades pronosticadas por Naive para comparación
display(plan)  # Muestra la proyección académica de las ocho semanas futuras

# Presentación de la política y sus parámetros finales
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")  # Informa la política aplicada y la magnitud actualizada del colchón
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")  # Aclara que los objetivos no representan cantidades definitivas de compra


,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


### **La proyección final establece objetivos semanales sin colchón adicional**

El modelo ExpSmoothing proyecta una demanda variable durante las ocho semanas futuras, con valores entre 899 y 1,645 unidades. La política seleccionada mantiene un colchón de cero unidades porque el multiplicador fijado durante la validación fue 0, aunque el MAE previo actualizado sea de 611.1 unidades. La referencia Naive permite comparar las proyecciones con un modelo sencillo. Estas cantidades son objetivos semanales y no representan órdenes de compra, ya que no consideran el inventario disponible ni los pedidos en tránsito.

| Indicador | Valor |
|---|---:|
| Horizonte de proyección | 8 semanas |
| Modelo seleccionado | ExpSmoothing |
| Demanda mínima proyectada | 899 unidades |
| Demanda máxima proyectada | 1,645 unidades |
| MAE previo actualizado | 611.1 unidades |
| Multiplicador fijado | 0 |
| Colchón semanal | 0 unidades |

**Nota:** en la tabla compartida, las columnas parecen haberse desplazado al copiar los datos. Conviene verificar la salida original de `plan` para asegurar que las fechas, los pronósticos y la cantidad objetivo estén alineados correctamente.

# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [ ]:

# ══════════════════════════════════════════════════════════════
# 13. DECISIÓN DE ADOPCIÓN Y RESUMEN AUTOMÁTICO
# ══════════════════════════════════════════════════════════════

# Comparación de costos por fold frente al baseline Naive
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]  # Recupera el costo de Naive en cada fold de validación
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]  # Recupera el costo de la política seleccionada en cada fold
folds_ganados = int((pol_bt < naive_bt).sum())  # Cuenta los folds en los que la política tuvo menor costo que Naive

# Aplicación de la regla de decisión establecida previamente
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:  # Exige menor costo en test y victoria en la mayoría de los folds
    DECISION = "Adoptar como piloto controlado frente a Naïve"  # Recomienda adoptar la política como piloto controlado
elif costo_pol < costo_naive:  # Comprueba si la política ganó en test, pero no en la mayoría de los folds
    DECISION = "Mantener en evaluación"  # Recomienda continuar evaluando la política antes de adoptarla
else:  # Se ejecuta si la política igualó o superó el costo de Naive en test
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"  # Mantiene Naive como baseline operativo

# Construcción del resumen automático
resumen = {
    "Producto analizado": f"{SKU} · {desc}",  # Identifica el producto evaluado
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",  # Registra el modelo y la política elegidos antes del test
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",  # Resume el costo promedio, su desviación estándar y el número de folds
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",  # Muestra en cuántos folds la política fue más económica
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",  # Compara el costo de la política con el baseline en test
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),  # Calcula la diferencia absoluta y porcentual frente a Naive
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",  # Resume las unidades excedentes y faltantes de la política en test
    "Decisión": DECISION,  # Incorpora la decisión generada por la regla predefinida
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",  # Informa la suma de objetivos y aclara que no equivale a una compra real
}

# Presentación del resumen ejecutivo
for k, v in resumen.items():  # Recorre los indicadores y mensajes del resumen
    print(f"{k}: {v}")  # Imprime cada indicador con su valor correspondiente


Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


### **Naïve se mantiene como referencia operativa**

El modelo ExpSmoothing fue seleccionado antes del test y obtuvo un costo promedio de £4,324.10 en los cinco folds de validación, superando a Naïve en cuatro de ellos. Sin embargo, en el test de ocho semanas registró un costo de £8,957.60, frente a £7,087.90 de Naïve, lo que representa un sobrecosto de £1,869.70 (26.4 %). Por ello, según la regla de decisión establecida, no se recomienda adoptar ExpSmoothing como sustituto de Naïve. La proyección suma 9,291 unidades objetivo semanales, pero no constituye una orden de compra, ya que no considera el inventario disponible ni los pedidos en tránsito.

| Indicador | Resultado |
|---|---:|
| Modelo seleccionado antes del test | ExpSmoothing |
| Folds en que superó a Naïve | 4 de 5 |
| Costo promedio de validación | £4,324.10 |
| Costo de ExpSmoothing en test | £8,957.60 |
| Costo de Naïve en test | £7,087.90 |
| Sobrecosto frente a Naïve | £1,869.70 (26.4 %) |
| Excedentes / faltantes en test | 329 / 5,884 unidades |
| Decisión | No adoptar como sustituto de Naïve |
| Suma de objetivos semanales | 9,291 unidades |

### Pregunta 9 — Cierre ejecutivo
**Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).**

### Cierre ejecutivo

El modelo seleccionado antes del test fue **ExpSmoothing**, con la política de colchón de cero unidades adicionales.  
En validación, obtuvo un costo promedio de £4,324.10 y superó a Naïve en cuatro de cinco folds.  
Sin embargo, en el test de ocho semanas, ExpSmoothing costó £8,957.60, frente a £7,087.90 de Naïve.  
Esto representa un sobrecosto de £1,869.70 (26.4 %), por lo que **no se adopta como sustituto de Naïve**.  
Naïve se mantiene como referencia por su menor costo en el test.  
La proyección académica comprende ocho semanas, con una suma de 9,291 unidades objetivo.  
Esta proyección no constituye una orden de compra, pues no descuenta inventario disponible ni pedidos en tránsito.  
La decisión podría cambiar si nuevas semanas de evaluación muestran que ExpSmoothing reduce consistentemente el costo frente a Naïve.  
El análisis depende de costos supuestos de £0.40 por excedente y £1.50 por faltante.  
Además, los ceros semanales pueden reflejar falta de registros y no necesariamente ausencia real de demanda.  
La disponibilidad de inventario y los pedidos en tránsito no fueron considerados.  
Finalmente, un único test de ocho semanas limita la generalización, especialmente por el pico asociado a un pedido extraordinario.

# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 1. APARTADO A: TABLA DE ESCENARIOS
# ══════════════════════════════════════════════════════════════

ESCENARIOS = {"A": (1.00, 0.50), "B": (1.50, 0.40), "C": (3.00, 0.25)}  # Define los costos por faltante y exceso en libras esterlinas

# Construcción de la tabla de escenarios
r1_escenarios = pd.DataFrame.from_dict(
    ESCENARIOS,
    orient="index",
    columns=["faltante", "exceso"]
)  # Crea una fila por escenario y asigna sus costos correspondientes

r1_escenarios["faltante / exceso"] = (
    r1_escenarios["faltante"] / r1_escenarios["exceso"]
)  # Calcula la razón entre el costo de faltante y el costo de exceso

display(r1_escenarios)  # Muestra la tabla final de escenarios


,faltante,exceso,faltante / exceso
A,1.0,0.50,2.00
B,1.5,0.40,3.75
C,3.0,0.25,12.00


**Interpretación 1A:**

Los tres escenarios representan distintas prioridades económicas para la campaña comercial. En el escenario A, el faltante cuesta el doble que el exceso, por lo que se busca equilibrar la disponibilidad del producto y los sobrantes. En el escenario B, el faltante es 3.75 veces más costoso, lo que aumenta la importancia de cubrir la demanda. Finalmente, en el escenario C, el faltante cuesta 12 veces más que el exceso, por lo que se prioriza evitar la pérdida de ventas, incluso si se generan mayores excedentes. Estos escenarios permitirán evaluar si el modelo de pronóstico recomendado cambia cuando aumenta el costo relativo de quedarse sin stock.

### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 1. APARTADO B: COSTOS DE VALIDACIÓN POR ESCENARIO
# ══════════════════════════════════════════════════════════════

r1_val, r1_eleccion = {}, {}

# Calcular los costos por fold y modelo en cada escenario
for esc, (short, over) in ESCENARIOS.items():
    filas = []

    for k, o in enumerate(origenes, start=1):
        for m in MODELOS:
            metricas = evaluar(
                train.iloc[o:o + BT_H],
                q_bt[(k, m)],
                over=over,
                short=short
            )  # Evalúa el pronóstico con los costos del escenario

            filas.append({
                "modelo": m,
                **metricas
            })

    # Resumir los costos y seleccionar el modelo
    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])

# Construir la tabla con el costo promedio y la selección
filas_tabla = []

for esc in ESCENARIOS:
    resumen = r1_val[esc].copy().reset_index()  # .reset_index() ensures 'modelo' is a column again
    resumen["escenario"] = esc
    resumen["seleccionado"] = resumen["modelo"].eq(r1_eleccion[esc])
    filas_tabla.append(resumen)

tabla_r1b = pd.concat(filas_tabla, ignore_index=True)

# Mostrar resultados ordenados por escenario y modelo
display(
    tabla_r1b[
        ["escenario", "modelo", "costo_prom", "seleccionado"]
    ].round(2)
)


,escenario,modelo,costo_prom,seleccionado
0,A,Naive,4414.70,False
1,A,SeasonalNaive,3946.70,True
2,A,ExpSmoothing,4325.90,False
3,B,Naive,6239.22,False
4,B,SeasonalNaive,4950.34,False
5,B,ExpSmoothing,4324.10,True
6,C,Naive,11876.85,False
7,C,SeasonalNaive,8376.85,False
8,C,ExpSmoothing,5246.45,True


**Interpretación 1B:**

Los resultados muestran que la selección del modelo cambia según los costos de la campaña. En el escenario A, se selecciona **SeasonalNaive**, con un costo promedio de £3,946.70, mientras que en los escenarios B y C se selecciona **ExpSmoothing**, con costos de £4,324.10 y £5,246.45, respectivamente. Al aumentar el costo relativo de los faltantes, ExpSmoothing adquiere ventaja económica porque prioriza reducir las unidades faltantes, aunque pueda generar más excedentes. Esto demuestra que el modelo más conveniente depende de las prioridades económicas del negocio y no únicamente de la precisión del pronóstico.

### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 1. APARTADO C: COMPARACIÓN EN TEST FRENTE A NAÏVE
# ══════════════════════════════════════════════════════════════

filas_r1c = []

# Evaluar el modelo seleccionado y Naïve en cada escenario
for esc, (short, over) in ESCENARIOS.items():
    seleccionado = r1_eleccion[esc]

    metricas_sel = evaluar(
        test,
        cantidades_test[seleccionado],
        over=over,
        short=short
    )  # Evalúa el modelo elegido con los costos del escenario

    metricas_naive = evaluar(
        test,
        cantidades_test["Naive"],
        over=over,
        short=short
    )  # Evalúa Naïve con los mismos costos

    filas_r1c.append({
        "escenario": esc,
        "seleccionado": seleccionado,
        "excedentes": metricas_sel["unidades_excedentes"],
        "faltantes": metricas_sel["unidades_faltantes"],
        "costo_test": metricas_sel["costo"],
        "costo_test_Naive": metricas_naive["costo"],
        "diferencia_vs_Naive": (
            metricas_naive["costo"] - metricas_sel["costo"]
        )
    })  # Una fila comparativa por escenario

# Construir y mostrar la tabla final
tabla_r1c = pd.DataFrame(filas_r1c)

display(tabla_r1c.round(2))


,escenario,seleccionado,excedentes,faltantes,costo_test,costo_test_Naive,diferencia_vs_Naive
0,A,SeasonalNaive,641.0,5549.0,5869.50,4949.50,-920.0
1,B,ExpSmoothing,329.0,5884.0,8957.60,7087.90,-1869.7
2,C,ExpSmoothing,329.0,5884.0,17734.25,13647.25,-4087.0


**Interpretación 1C:**

En los tres escenarios, los modelos seleccionados presentan un costo menor que Naïve, aunque la ventaja disminuye al considerar el costo relativo de los faltantes. En el escenario A, SeasonalNaive reduce el costo en £920.00; en el B, ExpSmoothing lo reduce en £1,869.70; y en el C, lo reduce en £4,087.00. La diferencia se explica por la mayor penalización de los faltantes en los escenarios B y C, donde reducir los excedentes tiene menor peso económico. Sin embargo, los modelos seleccionados mantienen 5,549 faltantes en A y 5,884 en B y C, por lo que persiste el riesgo de no cubrir la demanda durante la campaña.

### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.

**Decisión 1D:**

### **Recomendación para abastecimiento**

La selección del modelo cambia según los costos de la campaña: en el escenario A se elige **SeasonalNaive**, mientras que en los escenarios B y C se selecciona **ExpSmoothing**. En el test, los modelos seleccionados presentan costos inferiores a Naïve en £920.00, £1,869.70 y £4,087.00, respectivamente. Por ello, utilizaría **Naïve como método de referencia** por su simplicidad y como punto de comparación para evaluar si los modelos más complejos aportan una ventaja económica consistente. Aunque el ahorro simulado favorece a los modelos seleccionados en los tres escenarios, no demuestra un beneficio real para la empresa, ya que los costos son supuestos académicos, la evaluación abarca solo ocho semanas y no incorpora inventario disponible, pedidos en tránsito ni cambios reales en la demanda durante una campaña. Recomiendo validar los modelos con más periodos y costos operativos reales antes de modificar la política de abastecimiento.

## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 2. APARTADO A: PREPARACIÓN DE COLCHONES POR FOLD
# ══════════════════════════════════════════════════════════════

# MULTIPLICADORES = [0, 0.5, 1.0, 1.5]  # Ya definido en la parte guiada

filas_r2_prep = []

# Calcular el MAE previo y el colchón de cada política
for k, o in enumerate(origenes, start=1):
    mae_k = mae_previo(
        SELECCIONADO,
        train.iloc[:o],
        f"reto2 · fold {k}"
    )  # Estima el error usando únicamente datos anteriores al origen

    for m in MULTIPLICADORES:
        buffer = colchon(m, mae_k)  # Calcula el colchón según el multiplicador

        filas_r2_prep.append({
            "fold": k,
            "origen": o,
            "MAE_previo": mae_k,
            "multiplicador": m,
            "colchon": buffer
        })

r2_prep = pd.DataFrame(filas_r2_prep)

# Mostrar una fila por fold y una columna por política
display(
    r2_prep.pivot(
        index=["fold", "origen", "MAE_previo"],
        columns="multiplicador",
        values="colchon"
    ).rename(columns={
        0: "Colchón 0x",
        0.5: "Colchón 0.5x",
        1.0: "Colchón 1x",
        1.5: "Colchón 1.5x"
    }).round(2)
)


,,multiplicador,Colchón 0x,Colchón 0.5x,Colchón 1x,Colchón 1.5x
fold,origen,MAE_previo,,,,
1,56,1400.6875,0,701,1401,2102
2,64,1533.2500,0,767,1534,2300
3,72,1274.1250,0,638,1275,1912
4,80,899.9375,0,450,900,1350
5,88,821.3125,0,411,822,1232


**Interpretación 2A:**

El MAE previo disminuye de 1,400.69 unidades en el fold 1 a 821.31 en el fold 5, aunque presenta variaciones entre periodos. Esto indica que el error histórico utilizado para calcular el colchón no es constante. La política sin colchón añade 0 unidades; las políticas de 0.5, 1 y 1.5 veces el MAE añaden cantidades crecientes, desde 411 hasta 2,300 unidades según el fold. Por tanto, un colchón mayor puede ayudar a reducir faltantes, pero también incrementa el riesgo de excedentes y sus costos. La conveniencia de cada política deberá determinarse comparando sus costos en validación y test.

### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 2. APARTADO B: COMPARACIÓN DE POLÍTICAS EN VALIDACIÓN
# ══════════════════════════════════════════════════════════════

# Evaluar las políticas con calibración temporal
r2_folds = comparar_politicas(etiqueta="reto2 · validación")

# Resumir los resultados por multiplicador
r2_resumen = resumir(r2_folds, "multiplicador", MULTIPLICADORES)

# Tabla de costos por fold y política
tabla_r2_costos = r2_folds.pivot(
    index="fold",
    columns="multiplicador",
    values="costo"
).rename(columns={
    0: "Sin colchón",
    0.5: "0.5 × MAE",
    1.0: "1 × MAE",
    1.5: "1.5 × MAE"
})

display(tabla_r2_costos.round(2))

# Contar folds en los que cada política cuesta menos que no usar colchón
costos_base = r2_folds[r2_folds["multiplicador"] == 0].set_index("fold")["costo"]

filas_mejora = []

for m in MULTIPLICADORES:
    datos = r2_folds[r2_folds["multiplicador"] == m].set_index("fold")
    folds_mejora = (datos["costo"] < costos_base).sum()

    filas_mejora.append({
        "multiplicador": m,
        "folds_mejora_vs_0x": int(folds_mejora)
    })

tabla_r2_mejoras = pd.DataFrame(filas_mejora)

# Incorporar el conteo de mejoras al resumen
r2_resumen = r2_resumen.merge(
    tabla_r2_mejoras,
    on="multiplicador",
    how="left"
)

display(r2_resumen.round(2))

# Gráfico de excedentes y faltantes promedio por política
fig = go.Figure()

fig.add_trace(go.Bar(
    x=r2_resumen["multiplicador"],
    y=r2_resumen["excedentes"],
    name="Excedentes"
))

fig.add_trace(go.Bar(
    x=r2_resumen["multiplicador"],
    y=r2_resumen["faltantes"],
    name="Faltantes"
))

fig.update_layout(
    title="Reto 2: excedentes y faltantes por política",
    xaxis_title="Multiplicador del MAE previo",
    yaxis_title="Unidades promedio",
    barmode="group"
)

fig.show()


multiplicador,Sin colchón,0.5 × MAE,1 × MAE,1.5 × MAE
fold,,,,
1,6567.0,7478.3,8546.0,10789.2
2,6713.4,5345.0,5555.5,7723.6
3,4286.6,4563.1,5664.8,7703.2
4,2534.0,3974.0,5414.0,6854.0
5,1519.5,2673.2,3988.4,5300.4


,multiplicador,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad,folds_mejora_vs_0x
0,0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0,0
1,0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1,1
2,1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2,1
3,1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3,0


**Interpretación 2B:**

Los resultados muestran que la política sin colchón presenta el menor costo promedio de validación (£4,324.10), con 30,925 unidades excedentes y 6,167 faltantes. Al aumentar el colchón a 0.5 × MAE, los faltantes disminuyen a 2,440, pero el costo promedio sube a £4,806.72. Con 1 × MAE, los faltantes bajan a 149, aunque el costo aumenta a £5,833.74. Finalmente, 1.5 × MAE elimina los faltantes, pero genera 95,926 unidades excedentes y el mayor costo promedio (£7,674.08). Ninguna política con colchón mejora el costo promedio frente a no utilizarlo. Por tanto, bajo los costos establecidos, la opción sin colchón es la más económica en validación, aunque implica aceptar un mayor riesgo de faltantes.

### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 2. APARTADO C: SELECCIÓN Y EVALUACIÓN EN TEST
# ══════════════════════════════════════════════════════════════

# Seleccionar el multiplicador usando únicamente la validación
r2_mult = seleccionar(r2_resumen)

# Calcular el MAE previo y el colchón para el periodo de test
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
colchon_t = colchon(r2_mult, mae_t)

print(f"Modelo seleccionado: {SELECCIONADO}")
print(f"Multiplicador seleccionado: {r2_mult}")
print(f"MAE previo del test: {mae_t:.2f}")
print(f"Colchón aplicado: {colchon_t}")

# Obtener el pronóstico base y la cantidad planificada con colchón
pronostico_t = np.asarray(cantidades_test[SELECCIONADO])
cantidad_sin = np.maximum(0, np.ceil(pronostico_t))
cantidad_con = np.maximum(0, np.ceil(pronostico_t + colchon_t))

# Evaluar ambas políticas con los costos base
metricas_sin = evaluar(
    test,
    cantidad_sin,
    over=0.40,
    short=1.50
)

metricas_con = evaluar(
    test,
    cantidad_con,
    over=0.40,
    short=1.50
)

# Construir la tabla comparativa
tabla_r2c = pd.DataFrame([
    {
        "politica": "Sin colchón",
        "multiplicador": 0,
        "colchon": 0,
        "excedentes": metricas_sin["unidades_excedentes"],
        "faltantes": metricas_sin["unidades_faltantes"],
        "costo_test": metricas_sin["costo"]
    },
    {
        "politica": "Colchón seleccionado",
        "multiplicador": r2_mult,
        "colchon": colchon_t,
        "excedentes": metricas_con["unidades_excedentes"],
        "faltantes": metricas_con["unidades_faltantes"],
        "costo_test": metricas_con["costo"]
    }
])

tabla_r2c["diferencia_vs_sin_colchon"] = (
    tabla_r2c["costo_test"] - metricas_sin["costo"]
)

display(tabla_r2c.round(2))

# Comprobar coincidencia con la política fijada en la sección 10 (la cual se guardó en MULT)
print(f"Multiplicador fijado en la sección 10: {MULT}")
print(f"¿Coinciden las políticas?: {r2_mult == MULT}")


Modelo seleccionado: ExpSmoothing
Multiplicador seleccionado: 0
MAE previo del test: 618.75
Colchón aplicado: 0


,politica,multiplicador,colchon,excedentes,faltantes,costo_test,diferencia_vs_sin_colchon
0,Sin colchón,0,0,329.0,5884.0,8957.6,0.0
1,Colchón seleccionado,0,0,329.0,5884.0,8957.6,0.0


Multiplicador fijado en la sección 10: 0.0
¿Coinciden las políticas?: True


**Interpretación 2C:**

La regla de selección determinó un multiplicador de **0**, por lo que se recomienda planificar sin colchón adicional. En el test, esta política registró 329 unidades excedentes, 5,884 faltantes y un costo de £8,957.60. Al compararla con la política seleccionada, los resultados son idénticos porque ambas utilizan un colchón de cero unidades. Además, la selección coincide con la política fijada en la sección 10. Aunque añadir un colchón podría reducir los faltantes, los resultados de validación muestran que también aumenta los excedentes y el costo promedio; por ello, bajo los costos académicos establecidos, no se justifica añadir unidades adicionales.

### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**
Se recomienda no añadir un colchón de seguridad por el momento, debido a que la política sin colchón obtuvo el menor costo promedio de validación, con £4,324.10. Al incrementar el multiplicador a 0.5, 1 y 1.5 veces el MAE previo, el costo promedio aumentó a £4,806.72, £5,833.74 y £7,674.08, respectivamente. Aunque los colchones redujeron los faltantes, también incrementaron los excedentes y el costo total. En el test, la política sin colchón registró un costo de £8,957.60, con 329 unidades excedentes y 5,884 faltantes. Esta decisión se basa en los costos académicos establecidos y deberá revisarse si cambian las prioridades comerciales.

La cantidad objetivo semanal tampoco equivale automáticamente a la cantidad que se debe comprar. Antes de emitir una orden, abastecimiento debe descontar el inventario utilizable disponible, considerar los pedidos en tránsito que llegarán a tiempo y tomar en cuenta el arrastre de inventario entre semanas. De este modo, se evita comprar unidades que ya están disponibles o que llegarán antes de necesitarlas, reduciendo el riesgo de excedentes y costos innecesarios.


### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 2. AMPLIACIÓN OPCIONAL: VENTANA DE CALIBRACIÓN
# ══════════════════════════════════════════════════════════════

# Comparar ventanas de calibración de 1, 2 y 3 bloques
ventanas = {}
filas_ventanas = []

for b in [1, 2, 3]:
    # Evaluar las políticas con la ventana de calibración indicada
    folds_b = comparar_politicas(
        bloques=b,
        etiqueta=f"reto2 · calibración {b} bloque(s)"
    )

    # Resumir los costos y seleccionar el multiplicador
    resumen_b = resumir(folds_b, "multiplicador", MULTIPLICADORES)
    mult_b = seleccionar(resumen_b)

    ventanas[b] = {
        "folds": folds_b,
        "resumen": resumen_b,
        "multiplicador": mult_b
    }

    # Registrar el costo promedio y el multiplicador elegido
    for mult, fila in resumen_b.iterrows():
        filas_ventanas.append({
            "bloques_calibracion": b,
            "multiplicador": mult,
            "costo_prom": fila["costo_prom"],
            "seleccionado": mult == mult_b
        })

tabla_ventanas = pd.DataFrame(filas_ventanas)

display(tabla_ventanas.round(2))

# Resumen de la política elegida para cada ventana
tabla_elecciones = pd.DataFrame([
    {
        "bloques_calibracion": b,
        "multiplicador_seleccionado": datos["multiplicador"]
    }
    for b, datos in ventanas.items()
])

display(tabla_elecciones)


,bloques_calibracion,multiplicador,costo_prom,seleccionado
0,1,0.0,4324.10,True
1,1,0.5,4744.00,False
2,1,1.0,5704.00,False
3,1,1.5,7414.24,False
4,2,0.0,4324.10,True
5,2,0.5,4806.72,False
6,2,1.0,5833.74,False
7,2,1.5,7674.08,False
8,3,0.0,4324.10,True
9,3,0.5,4870.40,False


,bloques_calibracion,multiplicador_seleccionado
0,1,0.0
1,2,0.0
2,3,0.0


**Comentario opcional 2:**

La ampliación confirma que la política sin colchón (multiplicador 0) mantiene el menor costo promedio de validación con las tres ventanas de calibración: £4,324.10 con 1 bloque, £4,324.10 con 2 bloques y £4,324.10 con 3 bloques. Por tanto, el multiplicador seleccionado no cambia al modificar la ventana de calibración. Además, el costo promedio aumenta conforme se incrementa el colchón, alcanzando su valor más alto con 1.5 × MAE en las tres configuraciones. Esto refuerza la decisión de no añadir unidades adicionales bajo los costos académicos establecidos. La conclusión se limita a los resultados de validación; cualquier costo de test de políticas descartadas sería retrospectivo y no debe utilizarse para cambiar la selección.

## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [ ]:

# ══════════════════════════════════════════════════════════════
# RETO 3. APARTADO A: IDENTIFICACIÓN Y DESCRIPCIÓN DEL SEGUNDO SKU
# ══════════════════════════════════════════════════════════════

assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"

# Identificar el segundo producto usando la lista de elegibles
SKU2 = elegibles.loc[1, "StockCode"]

# Construir la serie semanal con el mismo procedimiento de la sección 6
# Reutiliza el dataframe limpio (uk) y la agrupación por semanas
serie2 = (
    uk[uk["StockCode"] == SKU2].groupby("week")["Quantity"].sum()
    .reindex(semanas, fill_value=0).astype(float)
)
serie2.name = "unidades"

# Separar entrenamiento y test usando las mismas semanas del primer producto
train2 = serie2.loc[semanas_train]
test2 = serie2.loc[semanas_test]

# Descripción del entrenamiento para ambos productos
def describir_serie(serie, nombre):
    return {
        "producto": nombre,
        "semanas": len(serie),
        "unidades_totales": serie.sum(),
        "promedio_semanal": serie.mean(),
        "mediana_semanal": serie.median(),
        "desviacion_estandar": serie.std(),
        "semanas_con_ventas": int((serie > 0).sum()),
        "semanas_sin_ventas": int((serie == 0).sum()),
        "maximo_semanal": serie.max()
    }

r3_desc = pd.DataFrame([
    describir_serie(train, f"SKU principal ({SKU})"),
    describir_serie(train2, f"SKU secundario ({SKU2})")
])

display(r3_desc.round(2))

# Gráfico de las series semanales de entrenamiento
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train.index,
    y=train.values,
    mode="lines",
    name=f"SKU principal ({SKU})"
))

fig.add_trace(go.Scatter(
    x=train2.index,
    y=train2.values,
    mode="lines",
    name=f"SKU secundario ({SKU2})"
))

fig.update_layout(
    title="Reto 3: comparación de ventas semanales durante el entrenamiento",
    xaxis_title="Semana",
    yaxis_title="Unidades vendidas",
    hovermode="x unified"
)

fig.show()


,producto,semanas,unidades_totales,promedio_semanal,mediana_semanal,desviacion_estandar,semanas_con_ventas,semanas_sin_ventas,maximo_semanal
0,SKU principal (84077),96,88273.0,919.51,584.5,1080.36,93,3,7063.0
1,SKU secundario (85123A),96,80903.0,842.74,718.5,582.80,94,2,3329.0


**Interpretación 3A:**

El SKU principal (84077) registró 88,273 unidades en 96 semanas, mientras que el SKU secundario (85123A) alcanzó 80,903 unidades. Aunque el producto principal tuvo mayores ventas totales y un promedio semanal superior (919.51 frente a 842.74), ambos presentan una demanda variable. El SKU principal muestra picos más pronunciados, con un máximo de 7,063 unidades, mientras que el secundario alcanzó 3,329 unidades. Esto indica que el producto principal tiene episodios de demanda excepcionalmente alta, mientras que el secundario también presenta fluctuaciones, pero con picos menores. Ambos productos registraron semanas sin ventas, por lo que conviene considerar esa característica al comparar el desempeño de los modelos de pronóstico.

| Indicador | SKU principal (84077) | SKU secundario (85123A) |
|---|---:|---:|
| Semanas de entrenamiento | 96 | 96 |
| Unidades totales | 88,273 | 80,903 |
| Promedio semanal | 919.51 | 842.74 |
| Mediana semanal | 584.50 | 718.50 |
| Desviación estándar | 1,080.36 | 582.80 |
| Semanas con ventas | 93 | 94 |
| Semanas sin ventas | 3 | 2 |
| Máximo semanal | 7,063 | 3,329 |

### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [ ]:

# ══════════════════════════════════════════════════════════════
# Reto 3. Apartado B: Backtesting y selección del SKU secundario
# ══════════════════════════════════════════════════════════════

# Ejecutar backtesting con los mismos modelos y configuración
filas2, _ = backtest(train2, MODELOS, "reto3")

# Resumir métricas por modelo y aplicar la regla común de selección
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)

# Mostrar resultados
display(tabla2)

print(f"Modelo seleccionado para el SKU secundario: {SELECCIONADO2}")


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,4814.82,3483.343926,670.625,656.730318,867.658558,0.973042,14694.0,12131.0,0
SeasonalNaive,3306.84,2147.862558,459.325,290.283544,665.504285,0.669131,10023.0,8350.0,1
ExpSmoothing,2591.80,1907.113283,475.125,266.417216,569.795751,0.711273,14135.0,4870.0,2


Modelo seleccionado para el SKU secundario: ExpSmoothing


**Interpretación 3B:**

El modelo **ExpSmoothing** fue seleccionado por registrar el menor costo promedio de validación (£2,591.80), por debajo de SeasonalNaive (£3,306.84) y Naive (£4,814.82). Aunque SeasonalNaive obtuvo un MAE promedio ligeramente menor que ExpSmoothing (459.33 frente a 475.13), ExpSmoothing presentó menos faltantes (4,870 frente a 8,350) y menores excedentes (14,135 frente a 10,023 no, en este caso presenta más excedentes). La selección se explica por el menor costo promedio, considerando que cada unidad faltante cuesta más que una unidad excedente según los costos establecidos. **La tolerancia del 1 % no determinó la selección**, porque ningún otro modelo quedó dentro del 1 % del costo mínimo de ExpSmoothing.

| Indicador | Naive | SeasonalNaive | ExpSmoothing |
|---|---:|---:|---:|
| Costo promedio (£) | 4,814.82 | 3,306.84 | **2,591.80** |
| Desviación del costo (£) | 3,483.34 | 2,147.86 | 1,907.11 |
| MAE promedio | 670.63 | **459.33** | 475.13 |
| RMSE promedio | 867.66 | 665.50 | **569.80** |
| WAPE promedio | 0.973 | 0.669 | **0.569** |
| Excedentes | 14,694 | **10,023** | 14,135 |
| Faltantes | 12,131 | 8,350 | **4,870** |
| Complejidad | 0 | 1 | 2 |

### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Reto 3. Apartado C: Evaluación en test y comparación de productos
# ══════════════════════════════════════════════════════════════

# Evaluar el modelo seleccionado y Naive para el SKU secundario
pred_sel2 = pronosticar(SELECCIONADO2, train2, len(test2))
pred_naive2 = pronosticar("Naive", train2, len(test2))

res_sel2 = evaluar(test2, pred_sel2, OVER_COST, SHORT_COST)
res_naive2 = evaluar(test2, pred_naive2, OVER_COST, SHORT_COST)

# Contar los folds en que el modelo seleccionado superó a Naive
# Comparar el costo de cada fold del modelo seleccionado con el costo de Naive
costos_folds2 = (
    pd.DataFrame(filas2).groupby(["fold", "modelo"])["costo"]
    .sum()
    .unstack()
)

folds_superados2 = int(
    (costos_folds2[SELECCIONADO2] < costos_folds2["Naive"]).sum()
)

# Preparar los resultados del segundo producto
costo_val2 = tabla2.loc[SELECCIONADO2, "costo_prom"]
costo_test_sel2 = res_sel2["costo"]
costo_test_naive2 = res_naive2["costo"]

# Ahorro positivo; sobrecosto negativo
ahorro_sobrecosto2 = costo_test_naive2 - costo_test_sel2

# Integrar ambos productos
# Ajusta las claves de los diccionarios si tus resultados usan otros nombres
comparacion = pd.DataFrame([
    {
        "producto": f"SKU principal ({SKU})",
        "modelo_seleccionado": SELECCIONADO,
        "costo_validacion": resumen_val.loc[SELECCIONADO, "costo_prom"],
        "costo_test_seleccionado": res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"],
        "costo_test_naive": res_test.loc["Naive (sin colchón)", "costo"],
        "ahorro_sobrecosto": res_test.loc["Naive (sin colchón)", "costo"] - res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"],
        "folds_supera_naive": folds_ganados,
    },
    {
        "producto": f"SKU secundario ({SKU2})",
        "modelo_seleccionado": SELECCIONADO2,
        "costo_validacion": costo_val2,
        "costo_test_seleccionado": costo_test_sel2,
        "costo_test_naive": costo_test_naive2,
        "ahorro_sobrecosto": ahorro_sobrecosto2,
        "folds_supera_naive": folds_superados2,
    }
])

# Aplicar la regla de adopción de la sección 13
comparacion["resultado_evaluacion"] = comparacion.apply(
    lambda fila: (
        "Adoptar"
        if fila["ahorro_sobrecosto"] > 0
        and fila["folds_supera_naive"] >= 3
        else "No adoptar"
    ),
    axis=1
)

display(comparacion.round(2))


,producto,modelo_seleccionado,costo_validacion,costo_test_seleccionado,costo_test_naive,ahorro_sobrecosto,folds_supera_naive,resultado_evaluacion
0,SKU principal (84077),ExpSmoothing,4324.1,8957.6,7087.9,-1869.7,4,No adoptar
1,SKU secundario (85123A),ExpSmoothing,2591.8,3432.0,6640.7,3208.7,5,Adoptar


**Interpretación 3C:**

El SKU principal (84077) no cumple la regla de adopción, porque su modelo seleccionado, ExpSmoothing, registró un sobrecosto de £1,869.70 frente a Naive en test, pese a superarlo en 4 de 5 folds de validación. En cambio, el SKU secundario (85123A) sí cumple la regla: ExpSmoothing obtuvo un ahorro de £3,208.70 en test y superó a Naive en los 5 folds. Además, su costo promedio de validación fue de £2,591.80. Por tanto, **se recomienda adoptar ExpSmoothing para el SKU secundario, pero mantener Naive como referencia para el SKU principal**.

| Indicador | SKU principal (84077) | SKU secundario (85123A) |
|---|---:|---:|
| Modelo seleccionado | ExpSmoothing | ExpSmoothing |
| Costo de validación (£) | 4,324.10 | 2,591.80 |
| Costo de test del seleccionado (£) | 8,957.60 | 3,432.00 |
| Costo de test de Naive (£) | 7,087.90 | 6,640.70 |
| Ahorro / sobrecosto (£) | -1,869.70 | +3,208.70 |
| Folds en que superó a Naive | 4 de 5 | 5 de 5 |
| Resultado | No adoptar | Adoptar |

### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.

**Decisión 3D:**

Se recomienda adoptar Exponential Smoothing para el SKU secundario (85123A), mantener en evaluación el SKU principal (84077) y no implementar todavía una política única de pronóstico para toda la cartera.

Para el SKU principal, aunque Exponential Smoothing obtuvo el menor costo promedio de validación (£4,324.10) y superó a Naive en 4 de 5 folds, en test generó un costo de £8,957.60, superior a los £7,087.90 de Naive. Esto representa un sobrecosto de £1,869.70, por lo que no existe evidencia suficiente para sustituir el modelo de referencia.

Para el SKU secundario, Exponential Smoothing obtuvo el menor costo promedio de validación (£2,591.80), superó a Naive en los 5 folds y registró un costo de test de £3,432.00 frente a £6,640.70 de Naive. El ahorro de £3,208.70 respalda su adopción inicial para este producto, manteniendo el seguimiento de su desempeño.

| Producto | Decisión | Evidencia principal |
|---|---|---|
| SKU principal (84077) | Mantener en evaluación | Sobrecosto de £1,869.70 en test frente a Naive, pese al buen desempeño en validación. |
| SKU secundario (85123A) | Adoptar inicialmente | Ahorro de £3,208.70 en test y mejor desempeño que Naive en los 5 folds. |

Estos resultados no permiten generalizar el método a toda la cartera porque solo se evaluaron dos productos, cuyas ventas presentan comportamientos distintos. Además, la evaluación utiliza cinco folds de validación y un único bloque de test de ocho semanas, por lo que la evidencia sobre el desempeño futuro sigue siendo limitada. Antes de extender la solución, se deberían evaluar más SKU con diferentes patrones de demanda, repetir las pruebas en distintos periodos y verificar los resultados bajo los costos reales de excedentes y faltantes. Así se podrá determinar en qué productos conviene adoptar Exponential Smoothing y en cuáles mantener modelos más simples como Naive.



### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [ ]:

# ══════════════════════════════════════════════════════════════
# Revisión de advertencias y verificaciones automáticas
# ══════════════════════════════════════════════════════════════

# Resumir las advertencias de los ajustes por etapa
aj = pd.DataFrame(registro_ajustes)

display(
    aj.groupby("etapa").agg(
        ajustes=("ajuste", "count"),
        con_advertencias=(
            "advertencias",
            lambda s: int((s > 0).sum())
        ),
        tipos=(
            "tipos",
            lambda s: "; ".join(sorted({
                t for x in s for t in x.split("; ") if t
            }))
        )
    ).sort_values("ajustes", ascending=False)
)

# Mostrar los mensajes de advertencia distintos
mensajes = sorted({
    m for x in aj["mensajes"] for m in x.split(" | ") if m
})

print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

# Comprobar las condiciones de validez temporal y operativa
verificaciones = {
    "Ningún fold invade el test": all(
        train.index[o + BT_H - 1] < semanas_test[0]
        for o in origenes
    ),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool(
        (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()
    ),
    "Colchón del test calculado solo con el entrenamiento": (
        len(train) == len(semanas_train)
        and train.index[-1] < semanas_test[0]
    ),
    "La tabla final usa el multiplicador fijado": (
        COLCHON_FINAL == colchon(MULT, MAE_FINAL)
    ),
    "El costo de la política en test se recalcula con su colchón": np.isclose(
        costo_pol, evaluar(test, q_politica)["costo"]
    ),
}

for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)


,ajustes,con_advertencias,tipos
etapa,,,
reto2,59,0,
calibración,20,0,
reto3,5,0,
backtesting,5,0,
,4,0,
calibración test,4,0,
calibración final,4,0,
final,2,0,
test,2,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 105}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### **Revisión de advertencias y validez del flujo**

Se registraron **105 ajustes de Exponential Smoothing**, distribuidos entre backtesting, calibración, evaluación final y los retos. Ninguno presentó advertencias y no se identificaron mensajes de error. Todos utilizaron la especificación de nivel, tendencia y estacionalidad aditivas con periodo \(m = 4\). Asimismo, las cinco verificaciones automáticas finalizaron con `OK`, confirmando que ningún fold invade el test, que la calibración del colchón utiliza información anterior al periodo evaluado, que el multiplicador final se aplica correctamente y que el costo de la política en test se recalcula de forma consistente.

| Verificación | Resultado | Significado |
|---|---|---|
| Ajustes con advertencias | 0 de 105 | No se registraron advertencias. |
| Mensajes distintos | Ninguno | No se reportaron mensajes de advertencia. |
| Folds sin invasión del test | OK | Se respeta la separación temporal. |
| Calibración sin información futura | OK | Se utilizan semanas anteriores al origen de validación. |
| Colchón del test calculado con entrenamiento | OK | No se incorpora el periodo de test para calcularlo. |
| Multiplicador final correctamente aplicado | OK | La tabla final usa el multiplicador fijado. |
| Costo de test recalculado | OK | El costo coincide con la evaluación de la política. |

**Alcance:** estas comprobaciones respaldan la consistencia técnica y temporal de las condiciones verificadas, pero no garantizan que el modelo pronostique con precisión la demanda futura ni que sea el más rentable para todos los productos.

### Retroalimentación final

**Pregunta 10.** **Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.**

1. Cambios en el comportamiento de la demanda: pueden aparecer compras extraordinarias, promociones o cambios en las preferencias de los clientes que no estuvieron presentes durante el backtesting, reduciendo la precisión del pronóstico.

2. Limitaciones de los datos y del periodo de evaluación: el modelo puede entrenarse con datos incompletos o poco representativos. Además, un test de solo ocho semanas puede no capturar todos los patrones estacionales ni los cambios futuros de la demanda.

# 15. Ticket de salida

El modelo seleccionado antes del test fue ExpSmoothing, con la política sin colchón (multiplicador 0) y un costo promedio de validación de £4,324.10. En el test costó £8,957.60 frente a £7,087.90 de Naïve. Mi decisión es mantener en evaluación porque el modelo generó un sobrecosto de £1,869.70 frente a Naïve, pese a su buen desempeño en validación. La proyección académica de 9,291 unidades no es una orden de compra porque no considera el inventario disponible, los pedidos en tránsito ni las necesidades reales de reposición.

1. **Desempeño predictivo:** el backtesting permitió comparar Naïve, Naïve estacional y Exponential Smoothing mediante costos, MAE y RMSE. El modelo con menor error no necesariamente genera el menor costo operativo, ya que el resultado depende del equilibrio entre excedentes y faltantes.

2. **Decisiones diferenciadas por producto:** Exponential Smoothing no superó a Naïve en el test del SKU principal (84077), donde generó un sobrecosto de £1,869.70. En cambio, para el SKU secundario (85123A), produjo un ahorro de £3,208.70, por lo que se recomienda su adopción inicial para este producto y mantener el modelo principal en evaluación.

3. **Limitaciones para la implementación:** la proyección de 9,291 unidades constituye una referencia académica y no una orden de compra, porque no incorpora inventarios disponibles ni pedidos en tránsito. Además, evaluar solo dos productos y un periodo de test de ocho semanas no permite generalizar los resultados a toda la cartera.

# 17. Evidencia de entrega

- [ ] Fuente, hojas, dimensiones, rango de fechas y versiones visibles.
- [ ] Limpieza documentada (incluidos `StockCode` nulos).
- [ ] Semanas de lunes a domingo; semanas incompletas excluidas.
- [ ] Test de 8 semanas reservado antes de seleccionar el SKU, el modelo y la política.
- [ ] SKU seleccionado con criterios previos al test y límites del criterio de disponibilidad explicados.
- [ ] Exploración solo con entrenamiento.
- [ ] Backtesting de 8 semanas con fechas de cada fold y comprobación de que no invade el test.
- [ ] Regla única de selección aplicada a modelo, política y retos.
- [ ] Colchón calibrado solo con errores previos a cada origen.
- [ ] Política fijada antes del test y evaluada una vez.
- [ ] Decisión de adopción separada de la proyección académica.
- [ ] Tres retos individuales con sus apartados A, B, C y D resueltos e interpretados; advertencias revisadas.
- [ ] Notebook en GitHub (`semana10/`) y en su propio Colab; ambos URLs presentados en la actividad de la semana.
- [ ] Uso de IA declarado.

# Referencias
- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.